In [1]:
# ============================================================
# 15_universities_v11.ipynb
# Domain: universities
# Goal: 95 high-quality RU/EN multihop benchmark queries for universities.
#
# v10 fixes compared to v9:
#   * removes trivial country-only L1 records from the default target;
#   * removes all user-facing "excluding / не включая" constraints;
#   * forbids redundant country+capital criteria in one query;
#   * reworks L3-L5 away from repeated Nobel-only templates;
#   * removes slow dynamic membership/admin pools from active L4-L5 generation;
#   * adds fast, diverse people/award/founder/named-after criteria;
#   * keeps constraints clean and human-readable; QIDs remain in metadata.


In [2]:
# ============================================================
from __future__ import annotations

from pathlib import Path
from dataclasses import asdict, fields
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence, Tuple
from collections import Counter
import json
import random
import re
import time


In [3]:
# ============================================================
# 0. Load shared project helpers


In [4]:
# ============================================================
# The project uses common_helpers.py as the executable version of
# 00_common_helpers.ipynb. Loading it preserves the exact BenchmarkExample
# dataclass and WikidataClient behavior used by other domains.
if "BenchmarkExample" not in globals():
    exec(Path("common_helpers.py").read_text(encoding="utf-8"), globals())

try:
    import pandas as pd
except Exception as e:
    raise RuntimeError("pandas is required by common_helpers.py and this notebook") from e

WD_CLIENT = globals().get("wd")
if WD_CLIENT is None or not hasattr(WD_CLIENT, "sparql_select"):
    raise RuntimeError("common_helpers.py did not initialize Wikidata client `wd` correctly")

print("✅ common helpers loaded; WD_CLIENT captured")


✅ Patched: WikidataClient.sparql_select (robust) + load_or_build_pool (safe)
✅ Patched: select_items_with_* используют ru/en fallback + repair_pool_labels чинит QID вместо label
✅ common helpers loaded; WD_CLIENT captured


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# ============================================================
# 1. Domain configuration


In [6]:
# ============================================================
DOMAIN = "universities"
VERSION = "v11"
Q_UNIVERSITY = "Q3918"  # university
Q_NOBEL_PRIZE = "Q7191"  # Nobel Prize

DOMAIN_OUTPUT_DIR = Path("out_wikidata_benchmark/domain_outputs")
DOMAIN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = DOMAIN_OUTPUT_DIR / "universities2.jsonl"

# User-requested distribution: 95 records total.
# L3+ are intentionally overrepresented because this domain is for multihop benchmarking.
TARGET_PER_LEVEL: Dict[str, int] = {
    # L1 country-only records were too easy for this multihop benchmark,
    # so v8 does not generate them by default.
    "L1": 0,
    "L2": 15,
    "L3": 25,
    "L4": 25,
    "L5": 30,
}
LEVELS: List[str] = ["L1", "L2", "L3", "L4", "L5"]

REQUESTED_BY_LEVEL: Dict[str, int] = {
    "L1": 5,
    "L2": 5,
    "L3": 4,
    "L4": 3,
    "L5": 3,
}

MAX_GOLD_BY_LEVEL: Dict[str, int] = {
    "L1": 220,
    "L2": 180,
    "L3": 140,
    "L4": 110,
    "L5": 90,
}

# Direct P31=university gives much cleaner gold than P31/P279*=university.
# It avoids many faculties, departments, schools, campuses and other higher-education
# related entities that are not natural answers to "universities".
DIRECT_INSTANCE_ONLY = True

SEED = 20260614
MAX_ATTEMPTS_PER_LEVEL = 4500
OVERWRITE_OUTPUT = True
RUN_GENERATION = True
RUN_FINAL_VALIDATION = True
DEBUG_GENERATOR_ERRORS = False
DEBUG_REJECTIONS = False

print(f"✅ config ready [{DOMAIN} {VERSION}] target={TARGET_PER_LEVEL}, direct_instance_only={DIRECT_INSTANCE_ONLY}")


✅ config ready [universities v11] target={'L1': 0, 'L2': 15, 'L3': 25, 'L4': 25, 'L5': 30}, direct_instance_only=True


In [7]:
# ============================================================
# 2. Schema and cleaning helpers


In [8]:
# ============================================================
EXPECTED_KEYS: List[str] = [f.name for f in fields(BenchmarkExample)]
CONTROL_RE = re.compile(r"[\u200e\u200f\u202a-\u202e\ufeff]")
CYRILLIC_RE = re.compile(r"[А-Яа-яЁё]")
QID_ONLY_RE = re.compile(r"^Q\d+$")
PID_ONLY_RE = re.compile(r"^P\d+$")
CONSTRAINT_META_KEY_RE = re.compile(r"(^|_)(qid|pid|wikidata|wd|sparql)($|_)", re.I)
BAD_QUERY_PHRASES_RE = re.compile(r"Wikidata|official website|coordinate locations?|координат|официальн\w+ сайт|викидан|не включая|исключая|excluding|exclude_anchor", re.I)

LAST_REJECT_REASON = ""
_GOLD_CACHE: Dict[str, Tuple[str, List[str], List[str], List[str], bool]] = {}


def clean_string(value: Any) -> str:
    value = CONTROL_RE.sub("", str(value or ""))
    value = re.sub(r"\s+", " ", value).strip()
    return value


def clean_text(value: Any) -> Any:
    if isinstance(value, str):
        return clean_string(value)
    if isinstance(value, list):
        return [clean_text(v) for v in value]
    if isinstance(value, tuple):
        return [clean_text(v) for v in value]
    if isinstance(value, dict):
        return {str(k): clean_text(v) for k, v in value.items() if v is not None}
    return value


def has_cyrillic(value: Any) -> bool:
    if isinstance(value, str):
        return bool(CYRILLIC_RE.search(value))
    if isinstance(value, list):
        return any(has_cyrillic(v) for v in value)
    if isinstance(value, dict):
        return any(has_cyrillic(k) or has_cyrillic(v) for k, v in value.items())
    return False


def has_qid_or_pid(value: Any) -> bool:
    if isinstance(value, str):
        s = clean_string(value)
        return bool(QID_ONLY_RE.fullmatch(s) or PID_ONLY_RE.fullmatch(s) or re.search(r"\b[QP]\d+\b", s))
    if isinstance(value, list):
        return any(has_qid_or_pid(v) for v in value)
    if isinstance(value, dict):
        return any(has_qid_or_pid(k) or has_qid_or_pid(v) for k, v in value.items())
    return False


def qid_from_any(value: Any) -> Optional[str]:
    try:
        return uri_to_qid(str(value or ""))
    except Exception:
        m = re.search(r"Q\d+", str(value or ""))
        return m.group(0) if m else None


def label_ok(label: Any) -> bool:
    label = clean_string(label)
    if not label:
        return False
    if QID_ONLY_RE.fullmatch(label) or PID_ONLY_RE.fullmatch(label):
        return False
    return True


def first_good_label(*labels: Any) -> str:
    for label in labels:
        if label_ok(label):
            return clean_string(label)
    return ""


def ru_name(label_ru: Any, label_en: Any = "") -> str:
    return first_good_label(label_ru, label_en)


def en_name(label_en: Any, label_ru: Any = "") -> str:
    return first_good_label(label_en, label_ru)


def ordered_as_benchmark_example(ex: BenchmarkExample) -> Dict[str, Any]:
    return asdict(ex)


def constraints_are_clean(constraints: Dict[str, Any]) -> bool:
    if not isinstance(constraints, dict):
        return False

    # Avoid the redundant pattern criticized in v7: deriving the country from
    # an anchor university and also restating the capital of that same country.
    # Each query should have at most one user-facing country/city criterion.
    if "country_capital" in constraints and ("country" in constraints or "country_from_university" in constraints):
        return False

    for key, value in constraints.items():
        if not isinstance(key, str):
            return False
        if "exclude" in key.lower():
            return False
        if CONSTRAINT_META_KEY_RE.search(key):
            return False
        if has_cyrillic(key) or has_cyrillic(value):
            return False
        if isinstance(value, dict):
            return False
        if has_qid_or_pid(key) or has_qid_or_pid(value):
            return False
        if isinstance(value, str) and BAD_QUERY_PHRASES_RE.search(value):
            return False
    return True


def reject(reason: str) -> None:
    global LAST_REJECT_REASON
    LAST_REJECT_REASON = reason
    if DEBUG_REJECTIONS:
        print("[reject]", reason)


def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        return []
    rows: List[Dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise RuntimeError(f"Bad JSONL line {line_no} in {path}: {e}") from e
    return rows


def append_example_jsonl(path: Path, ex: BenchmarkExample) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(ordered_as_benchmark_example(ex), ensure_ascii=False) + "\n")


def ru_university_word(k: int) -> str:
    k = abs(int(k))
    if 10 <= k % 100 <= 20:
        return "университетов"
    if k % 10 == 1:
        return "университет"
    if 2 <= k % 10 <= 4:
        return "университета"
    return "университетов"


def question_text_ok(text: str) -> bool:
    return bool(clean_string(text)) and BAD_QUERY_PHRASES_RE.search(text) is None

print("✅ schema helpers ready; clean constraints forbid qid/pid/wikidata/property-existence fields")


✅ schema helpers ready; clean constraints forbid qid/pid/wikidata/property-existence fields


In [9]:
# ============================================================
# 3. SPARQL builders


In [10]:
# ============================================================

def wd_entity(qid: str) -> str:
    qid = clean_string(qid)
    if not re.fullmatch(r"Q\d+", qid):
        raise ValueError(f"Not a QID: {qid}")
    return f"wd:{qid}"


def class_membership_line(item_var: str = "item", direct_only: bool = DIRECT_INSTANCE_ONLY) -> str:
    if direct_only:
        return f"?{item_var} wdt:P31 wd:{Q_UNIVERSITY} ."
    return f"?{item_var} wdt:P31/wdt:P279* wd:{Q_UNIVERSITY} ."


def label_block(item_var: str = "item") -> str:
    return f'''OPTIONAL {{ ?{item_var} rdfs:label ?{item_var}LabelRu FILTER(LANG(?{item_var}LabelRu) = "ru") . }}
      OPTIONAL {{ ?{item_var} rdfs:label ?{item_var}LabelEn FILTER(LANG(?{item_var}LabelEn) = "en") . }}
      FILTER(BOUND(?{item_var}LabelRu) || BOUND(?{item_var}LabelEn)) .'''


def dedupe_where_lines(where_lines: Sequence[str]) -> List[str]:
    out: List[str] = []
    seen = set()
    for line in where_lines:
        line = str(line).rstrip()
        if not line or line in seen:
            continue
        seen.add(line)
        out.append(line)
    return out


def build_select_sparql(where_lines: Sequence[str], *, item_var: str = "item", limit: int = 100, direct_only: bool = DIRECT_INSTANCE_ONLY) -> str:
    where = "\n      ".join(dedupe_where_lines(where_lines))
    return f'''
    SELECT DISTINCT ?{item_var} ?{item_var}LabelRu ?{item_var}LabelEn WHERE {{
      {class_membership_line(item_var=item_var, direct_only=direct_only)}
      {where}
      {label_block(item_var=item_var)}
    }}
    LIMIT {int(limit)}
    '''.strip()


def build_ask_sparql(where_lines: Sequence[str], *, item_var: str = "item", direct_only: bool = DIRECT_INSTANCE_ONLY) -> str:
    where = "\n      ".join(dedupe_where_lines(where_lines))
    return f'''
    ASK WHERE {{
      BIND(wd:{{ITEM}} AS ?{item_var})
      {class_membership_line(item_var=item_var, direct_only=direct_only)}
      {where}
    }}
    '''.strip()


def year_filter_lines(*, min_year: Optional[int] = None, max_year: Optional[int] = None, var: str = "inception", year_var: str = "year") -> List[str]:
    lines = [
        f"?item wdt:P571 ?{var} .",
        f"BIND(YEAR(?{var}) AS ?{year_var}) .",
    ]
    if min_year is not None and max_year is not None:
        lines.append(f"FILTER(?{year_var} >= {int(min_year)} && ?{year_var} <= {int(max_year)}) .")
    elif min_year is not None:
        lines.append(f"FILTER(?{year_var} >= {int(min_year)}) .")
    elif max_year is not None:
        lines.append(f"FILTER(?{year_var} <= {int(max_year)}) .")
    return lines


def inception_window_lines(y1: int, y2: int) -> List[str]:
    return year_filter_lines(min_year=int(y1), max_year=int(y2))


def before_year_lines(year: int) -> List[str]:
    return year_filter_lines(max_year=int(year) - 1)


def after_year_lines(year: int) -> List[str]:
    return year_filter_lines(min_year=int(year) + 1)


def exclude_qid_line(qid: str) -> str:
    return f"FILTER(?item != wd:{qid}) ."


def same_country_lines(anchor_qid: str, *, exclude_anchor: bool = True) -> List[str]:
    lines = [
        f"wd:{anchor_qid} wdt:P17 ?country .",
        "?item wdt:P17 ?country .",
    ]
    if exclude_anchor:
        lines.append(exclude_qid_line(anchor_qid))
    return lines


def same_admin_area_lines(anchor_qid: str, *, exclude_anchor: bool = True) -> List[str]:
    lines = [
        f"wd:{anchor_qid} wdt:P131 ?admin_area .",
        "?item wdt:P131 ?admin_area .",
    ]
    if exclude_anchor:
        lines.append(exclude_qid_line(anchor_qid))
    return lines


def same_membership_lines(anchor_qid: str, *, exclude_anchor: bool = True) -> List[str]:
    lines = [
        f"wd:{anchor_qid} wdt:P463 ?membership .",
        "?item wdt:P463 ?membership .",
    ]
    if exclude_anchor:
        lines.append(exclude_qid_line(anchor_qid))
    return lines

print("✅ SPARQL builders ready")


✅ SPARQL builders ready


In [11]:
# ============================================================
# 4. Gold collection and example construction


In [12]:
# ============================================================

def max_gold(level: str) -> int:
    return int(MAX_GOLD_BY_LEVEL[level])


def gold_query_limit(level: str) -> int:
    return max_gold(level) + 1


def rows_to_gold(rows: List[Dict[str, str]], *, item_var: str = "item") -> Tuple[List[str], List[str], List[str]]:
    qids: List[str] = []
    labels_ru: List[str] = []
    labels_en: List[str] = []
    seen = set()
    for row in rows:
        qid = qid_from_any(row.get(item_var))
        if not qid or qid in seen:
            continue
        label_ru = first_good_label(row.get(f"{item_var}LabelRu"), row.get(f"{item_var}LabelEn"))
        label_en = first_good_label(row.get(f"{item_var}LabelEn"), row.get(f"{item_var}LabelRu"))
        if not label_ok(label_ru) or not label_ok(label_en):
            continue
        seen.add(qid)
        qids.append(qid)
        labels_ru.append(label_ru)
        labels_en.append(label_en)
    return qids, labels_ru, labels_en


def collect_gold(where_lines: Sequence[str], *, level: str, item_var: str = "item") -> Tuple[str, List[str], List[str], List[str], bool]:
    """Collect exact gold with a tiny in-memory cache.

    v8 spent a lot of time re-running the same rejected/accepted WDQS queries
    during L4/L5 generation. The key is the normalized SELECT query; cached
    rows are copied before returning so downstream code cannot mutate cache.
    """
    sparql = build_select_sparql(where_lines, item_var=item_var, limit=gold_query_limit(level))
    cache_key = re.sub(r"\s+", " ", sparql).strip()
    cached = _GOLD_CACHE.get(cache_key)
    if cached is not None:
        _sparql, qids, labels_ru, labels_en, truncated = cached
        return _sparql, list(qids), list(labels_ru), list(labels_en), bool(truncated)

    data = WD_CLIENT.sparql_select(sparql)
    rows = rows_from_select(data)
    qids, labels_ru, labels_en = rows_to_gold(rows, item_var=item_var)
    truncated = len(qids) > max_gold(level)
    if truncated:
        qids = qids[:max_gold(level)]
        labels_ru = labels_ru[:max_gold(level)]
        labels_en = labels_en[:max_gold(level)]
    result = (sparql, list(qids), list(labels_ru), list(labels_en), bool(truncated))
    _GOLD_CACHE[cache_key] = result
    return result


def base_meta(template_id: str, constraint_entity_qids: Optional[Dict[str, Any]] = None, derived_labels: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    return clean_text({
        "source": "wikidata_wdqs",
        "generator_version": f"universities_{VERSION}",
        "template_id": template_id,
        "direct_instance_only": DIRECT_INSTANCE_ONLY,
        "constraint_entity_qids": constraint_entity_qids or {},
        "derived_labels": derived_labels or {},
        "note": "Entity identifiers used to execute SPARQL are stored here so constraints stay clean and human-readable.",
    })


def make_example(
    *,
    level: str,
    idx: int,
    query_text_ru: str,
    query_text_en: str,
    constraints: Dict[str, Any],
    where_lines: Sequence[str],
    template_id: str,
    template_family: str,
    is_advanced: bool,
    constraint_entity_qids: Optional[Dict[str, Any]] = None,
    derived_labels: Optional[Dict[str, Any]] = None,
) -> Optional[BenchmarkExample]:
    requested = int(REQUESTED_BY_LEVEL[level])
    constraints = clean_text(constraints)

    if not question_text_ok(query_text_ru) or not question_text_ok(query_text_en):
        reject(f"bad user-facing query text for {template_id}")
        return None
    if not constraints_are_clean(constraints):
        reject(f"bad constraints for {template_id}: {constraints}")
        return None

    try:
        sparql, qids, labels_ru, labels_en, truncated = collect_gold(where_lines, level=level)
    except Exception as e:
        reject(f"gold_query_failed:{template_id}:{e}")
        if DEBUG_GENERATOR_ERRORS:
            print(f"[WARN] gold query failed for {template_id}: {e}")
        return None

    if truncated:
        reject(f"truncated_gold:{template_id}:>{max_gold(level)}")
        return None
    if len(qids) < requested:
        reject(f"too_few_gold:{template_id}:{len(qids)}<{requested}")
        return None

    ask = build_ask_sparql(where_lines)
    if "wd:{ITEM}" not in ask:
        reject(f"bad_ask_validator:{template_id}")
        return None

    ex = BenchmarkExample(
        id=f"{DOMAIN}_{level.lower()}_{idx:04d}",
        domain=DOMAIN,
        complexity=level,
        query_text_ru=clean_text(query_text_ru),
        constraints=constraints,
        requested_count=requested,
        gold_answer_qids=qids,
        gold_answer_labels_ru=labels_ru,
        sparql_query=sparql,
        created_at=utc_now_z(),
        query_text_en=clean_text(query_text_en),
        gold_answer_labels_en=labels_en,
        is_advanced=bool(is_advanced),
        template_id=template_id,
        template_family=template_family,
        gold_truncated=False,
        ask_validator_sparql=ask,
        gold_collection_meta=base_meta(template_id, constraint_entity_qids, derived_labels),
    )

    if list(ordered_as_benchmark_example(ex).keys()) != EXPECTED_KEYS:
        reject(f"schema_mismatch:{template_id}")
        return None
    return ex

print("✅ gold collection ready; zero/underfilled/truncated gold records rejected")


✅ gold collection ready; zero/underfilled/truncated gold records rejected


In [13]:
# ============================================================
# 5. Static pools


In [14]:
# ============================================================
COUNTRIES: List[Dict[str, str]] = [
    {"qid": "Q30", "en": "United States", "ru": "США"},
    {"qid": "Q145", "en": "United Kingdom", "ru": "Великобритания"},
    {"qid": "Q183", "en": "Germany", "ru": "Германия"},
    {"qid": "Q142", "en": "France", "ru": "Франция"},
    {"qid": "Q159", "en": "Russia", "ru": "Россия"},
    {"qid": "Q17", "en": "Japan", "ru": "Япония"},
    {"qid": "Q148", "en": "China", "ru": "Китай"},
    {"qid": "Q16", "en": "Canada", "ru": "Канада"},
    {"qid": "Q38", "en": "Italy", "ru": "Италия"},
    {"qid": "Q29", "en": "Spain", "ru": "Испания"},
    {"qid": "Q55", "en": "Netherlands", "ru": "Нидерланды"},
    {"qid": "Q408", "en": "Australia", "ru": "Австралия"},
    {"qid": "Q39", "en": "Switzerland", "ru": "Швейцария"},
    {"qid": "Q40", "en": "Austria", "ru": "Австрия"},
    {"qid": "Q36", "en": "Poland", "ru": "Польша"},
    {"qid": "Q34", "en": "Sweden", "ru": "Швеция"},
    {"qid": "Q33", "en": "Finland", "ru": "Финляндия"},
    {"qid": "Q35", "en": "Denmark", "ru": "Дания"},
    {"qid": "Q27", "en": "Ireland", "ru": "Ирландия"},
    {"qid": "Q213", "en": "Czech Republic", "ru": "Чехия"},
    {"qid": "Q28", "en": "Hungary", "ru": "Венгрия"},
    {"qid": "Q41", "en": "Greece", "ru": "Греция"},
    {"qid": "Q43", "en": "Turkey", "ru": "Турция"},
    {"qid": "Q155", "en": "Brazil", "ru": "Бразилия"},
    {"qid": "Q96", "en": "Mexico", "ru": "Мексика"},
    {"qid": "Q414", "en": "Argentina", "ru": "Аргентина"},
    {"qid": "Q298", "en": "Chile", "ru": "Чили"},
    {"qid": "Q739", "en": "Colombia", "ru": "Колумбия"},
    {"qid": "Q668", "en": "India", "ru": "Индия"},
    {"qid": "Q884", "en": "South Korea", "ru": "Южная Корея"},
    {"qid": "Q865", "en": "Taiwan", "ru": "Тайвань"},
    {"qid": "Q212", "en": "Ukraine", "ru": "Украина"},
    {"qid": "Q232", "en": "Kazakhstan", "ru": "Казахстан"},
    {"qid": "Q219", "en": "Bulgaria", "ru": "Болгария"},
    {"qid": "Q218", "en": "Romania", "ru": "Румыния"},
    {"qid": "Q224", "en": "Croatia", "ru": "Хорватия"},
    {"qid": "Q215", "en": "Slovenia", "ru": "Словения"},
    {"qid": "Q214", "en": "Slovakia", "ru": "Словакия"},
    {"qid": "Q211", "en": "Latvia", "ru": "Латвия"},
    {"qid": "Q191", "en": "Estonia", "ru": "Эстония"},
    {"qid": "Q37", "en": "Lithuania", "ru": "Литва"},
    {"qid": "Q45", "en": "Portugal", "ru": "Португалия"},
    {"qid": "Q31", "en": "Belgium", "ru": "Бельгия"},
    {"qid": "Q32", "en": "Luxembourg", "ru": "Люксембург"},
    {"qid": "Q801", "en": "Israel", "ru": "Израиль"},
    {"qid": "Q258", "en": "South Africa", "ru": "Южная Африка"},
    {"qid": "Q117", "en": "Ghana", "ru": "Гана"},
    {"qid": "Q1033", "en": "Nigeria", "ru": "Нигерия"},
]


# Curated country-capital and anchor pools used by L3-L5.
# v6 used a large dynamic anchor query before L3; on real runs it could stall or
# fail before any L3+ records were written. v7 keeps those dynamic helpers as
# optional, but the active L3-L5 templates use this stable curated pool.
# QIDs stay in gold_collection_meta; constraints only expose human-readable labels.
CAPITAL_COUNTRIES: List[Dict[str, Any]] = [
    {"country_qid": "Q145", "country_en": "United Kingdom", "country_ru": "Великобритания", "capital_qid": "Q84", "capital_en": "London", "capital_ru": "Лондон"},
    {"country_qid": "Q55", "country_en": "Netherlands", "country_ru": "Нидерланды", "capital_qid": "Q727", "capital_en": "Amsterdam", "capital_ru": "Амстердам"},
    {"country_qid": "Q40", "country_en": "Austria", "country_ru": "Австрия", "capital_qid": "Q1741", "capital_en": "Vienna", "capital_ru": "Вена"},
    {"country_qid": "Q35", "country_en": "Denmark", "country_ru": "Дания", "capital_qid": "Q1748", "capital_en": "Copenhagen", "capital_ru": "Копенгаген"},
    {"country_qid": "Q33", "country_en": "Finland", "country_ru": "Финляндия", "capital_qid": "Q1757", "capital_en": "Helsinki", "capital_ru": "Хельсинки"},
    {"country_qid": "Q41", "country_en": "Greece", "country_ru": "Греция", "capital_qid": "Q1524", "capital_en": "Athens", "capital_ru": "Афины"},
    {"country_qid": "Q39", "country_en": "Switzerland", "country_ru": "Швейцария", "capital_qid": "Q70", "capital_en": "Bern", "capital_ru": "Берн"},
    {"country_qid": "Q29", "country_en": "Spain", "country_ru": "Испания", "capital_qid": "Q2807", "capital_en": "Madrid", "capital_ru": "Мадрид"},
    {"country_qid": "Q38", "country_en": "Italy", "country_ru": "Италия", "capital_qid": "Q220", "capital_en": "Rome", "capital_ru": "Рим"},
    {"country_qid": "Q16", "country_en": "Canada", "country_ru": "Канада", "capital_qid": "Q1930", "capital_en": "Ottawa", "capital_ru": "Оттава"},
    {"country_qid": "Q96", "country_en": "Mexico", "country_ru": "Мексика", "capital_qid": "Q1489", "capital_en": "Mexico City", "capital_ru": "Мехико"},
    {"country_qid": "Q258", "country_en": "South Africa", "country_ru": "Южная Африка", "capital_qid": "Q3926", "capital_en": "Pretoria", "capital_ru": "Претория"},
    {"country_qid": "Q215", "country_en": "Slovenia", "country_ru": "Словения", "capital_qid": "Q437", "capital_en": "Ljubljana", "capital_ru": "Любляна"},
    {"country_qid": "Q224", "country_en": "Croatia", "country_ru": "Хорватия", "capital_qid": "Q1435", "capital_en": "Zagreb", "capital_ru": "Загреб"},
    {"country_qid": "Q211", "country_en": "Latvia", "country_ru": "Латвия", "capital_qid": "Q1773", "capital_en": "Riga", "capital_ru": "Рига"},
    {"country_qid": "Q37", "country_en": "Lithuania", "country_ru": "Литва", "capital_qid": "Q216", "capital_en": "Vilnius", "capital_ru": "Вильнюс"},
]

STATIC_ANCHORS: List[Dict[str, Any]] = [
    {"qid": "Q35794", "en": "University of Cambridge", "ru": "Кембриджский университет", "country_qid": "Q145", "country_en": "United Kingdom", "country_ru": "Великобритания", "capital_qid": "Q84", "capital_en": "London", "capital_ru": "Лондон", "year": 1209},
    {"qid": "Q34433", "en": "University of Oxford", "ru": "Оксфордский университет", "country_qid": "Q145", "country_en": "United Kingdom", "country_ru": "Великобритания", "capital_qid": "Q84", "capital_en": "London", "capital_ru": "Лондон", "year": 1096},
    {"qid": "Q192775", "en": "University of Glasgow", "ru": "Университет Глазго", "country_qid": "Q145", "country_en": "United Kingdom", "country_ru": "Великобритания", "capital_qid": "Q84", "capital_en": "London", "capital_ru": "Лондон", "year": 1451},
    {"qid": "Q160302", "en": "University of Edinburgh", "ru": "Эдинбургский университет", "country_qid": "Q145", "country_en": "United Kingdom", "country_ru": "Великобритания", "capital_qid": "Q84", "capital_en": "London", "capital_ru": "Лондон", "year": 1582},
    {"qid": "Q156598", "en": "Leiden University", "ru": "Лейденский университет", "country_qid": "Q55", "country_en": "Netherlands", "country_ru": "Нидерланды", "capital_qid": "Q727", "capital_en": "Amsterdam", "capital_ru": "Амстердам", "year": 1575},
    {"qid": "Q214341", "en": "University of Amsterdam", "ru": "Амстердамский университет", "country_qid": "Q55", "country_en": "Netherlands", "country_ru": "Нидерланды", "capital_qid": "Q727", "capital_en": "Amsterdam", "capital_ru": "Амстердам", "year": 1632},
    {"qid": "Q850730", "en": "University of Groningen", "ru": "Университет Гронингена", "country_qid": "Q55", "country_en": "Netherlands", "country_ru": "Нидерланды", "capital_qid": "Q727", "capital_en": "Amsterdam", "capital_ru": "Амстердам", "year": 1614},
    {"qid": "Q165980", "en": "University of Vienna", "ru": "Венский университет", "country_qid": "Q40", "country_en": "Austria", "country_ru": "Австрия", "capital_qid": "Q1741", "capital_en": "Vienna", "capital_ru": "Вена", "year": 1365},
    {"qid": "Q622683", "en": "University of Graz", "ru": "Грацский университет", "country_qid": "Q40", "country_en": "Austria", "country_ru": "Австрия", "capital_qid": "Q1741", "capital_en": "Vienna", "capital_ru": "Вена", "year": 1585},
    {"qid": "Q186285", "en": "University of Copenhagen", "ru": "Копенгагенский университет", "country_qid": "Q35", "country_en": "Denmark", "country_ru": "Дания", "capital_qid": "Q1748", "capital_en": "Copenhagen", "capital_ru": "Копенгаген", "year": 1479},
    {"qid": "Q924265", "en": "Aarhus University", "ru": "Орхусский университет", "country_qid": "Q35", "country_en": "Denmark", "country_ru": "Дания", "capital_qid": "Q1748", "capital_en": "Copenhagen", "capital_ru": "Копенгаген", "year": 1928},
    {"qid": "Q28695", "en": "University of Helsinki", "ru": "Хельсинкский университет", "country_qid": "Q33", "country_en": "Finland", "country_ru": "Финляндия", "capital_qid": "Q1757", "capital_en": "Helsinki", "capital_ru": "Хельсинки", "year": 1640},
    {"qid": "Q501841", "en": "University of Turku", "ru": "Университет Турку", "country_qid": "Q33", "country_en": "Finland", "country_ru": "Финляндия", "capital_qid": "Q1757", "capital_en": "Helsinki", "capital_ru": "Хельсинки", "year": 1920},
    {"qid": "Q319078", "en": "University of Melbourne", "ru": "Мельбурнский университет", "country_qid": "Q408", "country_en": "Australia", "country_ru": "Австралия", "capital_qid": "Q3114", "capital_en": "Canberra", "capital_ru": "Канберра", "year": 1853},
    {"qid": "Q866012", "en": "University of Queensland", "ru": "Квинслендский университет", "country_qid": "Q408", "country_en": "Australia", "country_ru": "Австралия", "capital_qid": "Q3114", "capital_en": "Canberra", "capital_ru": "Канберра", "year": 1909},
    {"qid": "Q734764", "en": "University of New South Wales", "ru": "Университет Нового Южного Уэльса", "country_qid": "Q408", "country_en": "Australia", "country_ru": "Австралия", "capital_qid": "Q3114", "capital_en": "Canberra", "capital_ru": "Канберра", "year": 1949},
    {"qid": "Q547867", "en": "National and Kapodistrian University of Athens", "ru": "Афинский университет", "country_qid": "Q41", "country_en": "Greece", "country_ru": "Греция", "capital_qid": "Q1524", "capital_en": "Athens", "capital_ru": "Афины", "year": 1837},
    {"qid": "Q550263", "en": "University of Piraeus", "ru": "Университет Пирея", "country_qid": "Q41", "country_en": "Greece", "country_ru": "Греция", "capital_qid": "Q1524", "capital_en": "Athens", "capital_ru": "Афины", "year": 1938},
    {"qid": "Q222738", "en": "National Autonomous University of Mexico", "ru": "Национальный автономный университет Мексики", "country_qid": "Q96", "country_en": "Mexico", "country_ru": "Мексика", "capital_qid": "Q1489", "capital_en": "Mexico City", "capital_ru": "Мехико", "year": 1910},
    {"qid": "Q29716", "en": "Universidad Autónoma Nuevo León", "ru": "Автономный университет Нуэво-Леона", "country_qid": "Q96", "country_en": "Mexico", "country_ru": "Мексика", "capital_qid": "Q1489", "capital_en": "Mexico City", "capital_ru": "Мехико", "year": 1933},
]

STATIC_ANCHOR_PAIRS: List[Tuple[Dict[str, Any], Dict[str, Any]]] = [
    (STATIC_ANCHORS[0], STATIC_ANCHORS[2]),
    (STATIC_ANCHORS[0], STATIC_ANCHORS[3]),
    (STATIC_ANCHORS[4], STATIC_ANCHORS[5]),
    (STATIC_ANCHORS[7], STATIC_ANCHORS[8]),
    (STATIC_ANCHORS[9], STATIC_ANCHORS[10]),
    (STATIC_ANCHORS[11], STATIC_ANCHORS[12]),
    (STATIC_ANCHORS[13], STATIC_ANCHORS[15]),
    (STATIC_ANCHORS[16], STATIC_ANCHORS[17]),
    (STATIC_ANCHORS[18], STATIC_ANCHORS[19]),
]

# Extra capital/anchor rows make the Nobel-laureate templates productive and
# diversify L3-L5 beyond the smaller European pool. QIDs are metadata only.
EXTRA_CAPITAL_COUNTRIES: List[Dict[str, Any]] = [
    {"country_qid": "Q30", "country_en": "United States", "country_ru": "США", "capital_qid": "Q61", "capital_en": "Washington, D.C.", "capital_ru": "Вашингтон"},
    {"country_qid": "Q183", "country_en": "Germany", "country_ru": "Германия", "capital_qid": "Q64", "capital_en": "Berlin", "capital_ru": "Берлин"},
    {"country_qid": "Q142", "country_en": "France", "country_ru": "Франция", "capital_qid": "Q90", "capital_en": "Paris", "capital_ru": "Париж"},
    {"country_qid": "Q17", "country_en": "Japan", "country_ru": "Япония", "capital_qid": "Q1490", "capital_en": "Tokyo", "capital_ru": "Токио"},
    {"country_qid": "Q34", "country_en": "Sweden", "country_ru": "Швеция", "capital_qid": "Q1754", "capital_en": "Stockholm", "capital_ru": "Стокгольм"},
    {"country_qid": "Q36", "country_en": "Poland", "country_ru": "Польша", "capital_qid": "Q270", "capital_en": "Warsaw", "capital_ru": "Варшава"},
]
for _cc in EXTRA_CAPITAL_COUNTRIES:
    if _cc["country_qid"] not in {c["country_qid"] for c in CAPITAL_COUNTRIES}:
        CAPITAL_COUNTRIES.append(_cc)

EXTRA_STATIC_ANCHORS: List[Dict[str, Any]] = [
    {"qid": "Q13371", "en": "Harvard University", "ru": "Гарвардский университет", "country_qid": "Q30", "country_en": "United States", "country_ru": "США", "capital_qid": "Q61", "capital_en": "Washington, D.C.", "capital_ru": "Вашингтон", "year": 1636},
    {"qid": "Q49112", "en": "Yale University", "ru": "Йельский университет", "country_qid": "Q30", "country_en": "United States", "country_ru": "США", "capital_qid": "Q61", "capital_en": "Washington, D.C.", "capital_ru": "Вашингтон", "year": 1701},
    {"qid": "Q131252", "en": "University of Chicago", "ru": "Чикагский университет", "country_qid": "Q30", "country_en": "United States", "country_ru": "США", "capital_qid": "Q61", "capital_en": "Washington, D.C.", "capital_ru": "Вашингтон", "year": 1890},
    {"qid": "Q151510", "en": "Heidelberg University", "ru": "Гейдельбергский университет", "country_qid": "Q183", "country_en": "Germany", "country_ru": "Германия", "capital_qid": "Q64", "capital_en": "Berlin", "capital_ru": "Берлин", "year": 1386},
    {"qid": "Q152087", "en": "Humboldt University of Berlin", "ru": "Берлинский университет имени Гумбольдта", "country_qid": "Q183", "country_en": "Germany", "country_ru": "Германия", "capital_qid": "Q64", "capital_en": "Berlin", "capital_ru": "Берлин", "year": 1810},
    {"qid": "Q209842", "en": "University of Paris", "ru": "Парижский университет", "country_qid": "Q142", "country_en": "France", "country_ru": "Франция", "capital_qid": "Q90", "capital_en": "Paris", "capital_ru": "Париж", "year": 1150},
    {"qid": "Q7842", "en": "University of Tokyo", "ru": "Токийский университет", "country_qid": "Q17", "country_en": "Japan", "country_ru": "Япония", "capital_qid": "Q1490", "capital_en": "Tokyo", "capital_ru": "Токио", "year": 1877},
]
STATIC_ANCHORS.extend(EXTRA_STATIC_ANCHORS)
_anchor_by_qid = {a["qid"]: a for a in STATIC_ANCHORS}
for _a, _b in [("Q13371", "Q49112"), ("Q49112", "Q131252"), ("Q151510", "Q152087"), ("Q209842", "Q209842")]:
    if _a in _anchor_by_qid and _b in _anchor_by_qid and _a != _b:
        STATIC_ANCHOR_PAIRS.append((_anchor_by_qid[_a], _anchor_by_qid[_b]))

# Prefer countries where direct P31=university gold is not huge; this keeps
# complete gold lists instead of forcing broad/truncated records.
COUNTRIES = [c for c in COUNTRIES if c["qid"] in {cc["country_qid"] for cc in CAPITAL_COUNTRIES} | {"Q408"}]


def pick_static_anchor(rng: random.Random) -> Dict[str, Any]:
    return rng.choice(STATIC_ANCHORS)


def pick_static_anchor_pair(rng: random.Random) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    return rng.choice(STATIC_ANCHOR_PAIRS)


def pick_capital_country(rng: random.Random) -> Dict[str, Any]:
    return rng.choice(CAPITAL_COUNTRIES)


def capital_country_lines(country: Dict[str, Any]) -> List[str]:
    return [
        f"?item wdt:P17 ?country .",
        f"?country wdt:P36 wd:{country['capital_qid']} .",
    ]


def static_same_country_lines(anchor: Dict[str, Any], *, exclude_anchor: bool = True) -> List[str]:
    lines = [
        f"wd:{anchor['qid']} wdt:P17 ?country .",
        "?item wdt:P17 ?country .",
    ]
    if exclude_anchor:
        lines.append(exclude_qid_line(anchor["qid"]))
    return lines

print(f"✅ curated L3-L5 pools ready: anchors={len(STATIC_ANCHORS)}, capital countries={len(CAPITAL_COUNTRIES)}")

YEAR_WINDOWS_L2: List[Tuple[int, int]] = [
    (1200, 1799), (1800, 1849), (1850, 1899), (1900, 1924),
    (1925, 1949), (1950, 1979), (1980, 1999), (2000, 2026),
]
YEAR_WINDOWS_L3: List[Tuple[int, int]] = [
    (1200, 1899), (1800, 1949), (1850, 1949), (1900, 1979), (1950, 2026),
]
YEAR_WINDOWS_L4: List[Tuple[int, int]] = [
    (1200, 1799), (1800, 1899), (1850, 1949), (1900, 1949), (1950, 1979), (1980, 2026),
]
YEAR_WINDOWS_L5: List[Tuple[int, int]] = [
    (1800, 1949), (1850, 1949), (1900, 1979), (1950, 2026),
]
REFERENCE_YEARS: List[int] = [1600, 1700, 1800, 1850, 1900, 1950, 1980, 2000]

print(f"✅ static pools ready: countries={len(COUNTRIES)}")


✅ curated L3-L5 pools ready: anchors=27, capital countries=22
✅ static pools ready: countries=23


In [15]:
# ============================================================
# 6. Lazy anchor, membership and admin pools


In [16]:
# ============================================================
_ANCHOR_POOL_DF: Optional[pd.DataFrame] = None
_MEMBERSHIP_POOL_DF: Optional[pd.DataFrame] = None
_ADMIN_POOL_DF: Optional[pd.DataFrame] = None


def normalize_pool_df(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or len(df) == 0:
        return pd.DataFrame()
    out = df.copy()
    for col in out.columns:
        out[col] = out[col].fillna("").astype(str).map(clean_string)
    return out.drop_duplicates().reset_index(drop=True)


def build_anchor_pool() -> pd.DataFrame:
    # Direct P31=university keeps the anchor pool much cleaner than subclass closure.
    # Membership and admin fields are optional; templates requiring them filter later.
    sparql = f'''
    SELECT DISTINCT
      ?item ?itemLabelRu ?itemLabelEn
      ?country ?countryLabelRu ?countryLabelEn
      ?inception
      ?member ?memberLabelRu ?memberLabelEn
      ?admin ?adminLabelRu ?adminLabelEn
    WHERE {{
      ?item wdt:P31 wd:{Q_UNIVERSITY} .
      ?item wdt:P17 ?country .
      OPTIONAL {{ ?item wdt:P571 ?inception . }}
      OPTIONAL {{ ?item wdt:P463 ?member . }}
      OPTIONAL {{ ?item wdt:P131 ?admin . }}
      OPTIONAL {{ ?item rdfs:label ?itemLabelRu FILTER(LANG(?itemLabelRu) = "ru") . }}
      OPTIONAL {{ ?item rdfs:label ?itemLabelEn FILTER(LANG(?itemLabelEn) = "en") . }}
      OPTIONAL {{ ?country rdfs:label ?countryLabelRu FILTER(LANG(?countryLabelRu) = "ru") . }}
      OPTIONAL {{ ?country rdfs:label ?countryLabelEn FILTER(LANG(?countryLabelEn) = "en") . }}
      OPTIONAL {{ ?member rdfs:label ?memberLabelRu FILTER(LANG(?memberLabelRu) = "ru") . }}
      OPTIONAL {{ ?member rdfs:label ?memberLabelEn FILTER(LANG(?memberLabelEn) = "en") . }}
      OPTIONAL {{ ?admin rdfs:label ?adminLabelRu FILTER(LANG(?adminLabelRu) = "ru") . }}
      OPTIONAL {{ ?admin rdfs:label ?adminLabelEn FILTER(LANG(?adminLabelEn) = "en") . }}
      FILTER(BOUND(?itemLabelRu) || BOUND(?itemLabelEn)) .
    }}
    LIMIT 3500
    '''.strip()
    rows = rows_from_select(WD_CLIENT.sparql_select(sparql))
    records: List[Dict[str, Any]] = []
    for row in rows:
        item_qid = qid_from_any(row.get("item"))
        country_qid = qid_from_any(row.get("country"))
        member_qid = qid_from_any(row.get("member"))
        admin_qid = qid_from_any(row.get("admin"))
        item_en = en_name(row.get("itemLabelEn"), row.get("itemLabelRu"))
        item_ru = ru_name(row.get("itemLabelRu"), row.get("itemLabelEn"))
        country_en = en_name(row.get("countryLabelEn"), row.get("countryLabelRu"))
        country_ru = ru_name(row.get("countryLabelRu"), row.get("countryLabelEn"))
        member_en = en_name(row.get("memberLabelEn"), row.get("memberLabelRu"))
        member_ru = ru_name(row.get("memberLabelRu"), row.get("memberLabelEn"))
        admin_en = en_name(row.get("adminLabelEn"), row.get("adminLabelRu"))
        admin_ru = ru_name(row.get("adminLabelRu"), row.get("adminLabelEn"))
        inception_year = ""
        m = re.match(r"^(\d{3,4})", clean_string(row.get("inception", "")))
        if m:
            inception_year = m.group(1)
        if not item_qid or not country_qid or not item_en or not item_ru:
            continue
        records.append({
            "item_qid": item_qid,
            "item_label_ru": item_ru,
            "item_label_en": item_en,
            "country_qid": country_qid,
            "country_label_ru": country_ru,
            "country_label_en": country_en,
            "inception_year": inception_year,
            "member_qid": member_qid or "",
            "member_label_ru": member_ru,
            "member_label_en": member_en,
            "admin_qid": admin_qid or "",
            "admin_label_ru": admin_ru,
            "admin_label_en": admin_en,
        })
    return normalize_pool_df(pd.DataFrame(records))


def get_anchor_pool() -> pd.DataFrame:
    global _ANCHOR_POOL_DF
    if _ANCHOR_POOL_DF is None:
        _ANCHOR_POOL_DF = load_or_build_pool_safe("universities_anchor_pool_v8_direct", build_anchor_pool)
        _ANCHOR_POOL_DF = normalize_pool_df(_ANCHOR_POOL_DF)
        print(f"[pool] anchors: {len(_ANCHOR_POOL_DF)} rows")
    return _ANCHOR_POOL_DF


def build_membership_pool() -> pd.DataFrame:
    df = get_anchor_pool()
    if df is None or len(df) == 0 or "member_qid" not in df.columns:
        return pd.DataFrame()
    sub = df[df["member_qid"].str.fullmatch(r"Q\d+").fillna(False) & df["member_label_en"].ne("")].copy()
    if len(sub) == 0:
        return pd.DataFrame()
    counts = sub.groupby(["member_qid", "member_label_en", "member_label_ru"], dropna=False).size().reset_index(name="n")
    counts = counts[counts["n"] >= 3].copy()
    return normalize_pool_df(counts.sort_values(["n", "member_label_en"], ascending=[False, True]))


def get_membership_pool() -> pd.DataFrame:
    global _MEMBERSHIP_POOL_DF
    if _MEMBERSHIP_POOL_DF is None:
        _MEMBERSHIP_POOL_DF = load_or_build_pool_safe("universities_membership_pool_v8_direct", build_membership_pool)
        _MEMBERSHIP_POOL_DF = normalize_pool_df(_MEMBERSHIP_POOL_DF)
        print(f"[pool] memberships: {len(_MEMBERSHIP_POOL_DF)} rows")
    return _MEMBERSHIP_POOL_DF


def build_admin_pool() -> pd.DataFrame:
    df = get_anchor_pool()
    if df is None or len(df) == 0 or "admin_qid" not in df.columns:
        return pd.DataFrame()
    sub = df[df["admin_qid"].str.fullmatch(r"Q\d+").fillna(False) & df["admin_label_en"].ne("")].copy()
    if len(sub) == 0:
        return pd.DataFrame()
    counts = sub.groupby(["admin_qid", "admin_label_en", "admin_label_ru", "country_qid", "country_label_en", "country_label_ru"], dropna=False).size().reset_index(name="n")
    counts = counts[counts["n"] >= 3].copy()
    return normalize_pool_df(counts.sort_values(["n", "admin_label_en"], ascending=[False, True]))


def get_admin_pool() -> pd.DataFrame:
    global _ADMIN_POOL_DF
    if _ADMIN_POOL_DF is None:
        _ADMIN_POOL_DF = load_or_build_pool_safe("universities_admin_pool_v8_direct", build_admin_pool)
        _ADMIN_POOL_DF = normalize_pool_df(_ADMIN_POOL_DF)
        print(f"[pool] admin areas: {len(_ADMIN_POOL_DF)} rows")
    return _ADMIN_POOL_DF


def sample_df_row(df: pd.DataFrame, rng: random.Random) -> Optional[Dict[str, Any]]:
    if df is None or len(df) == 0:
        return None
    row = df.sample(1, random_state=rng.randint(0, 10**9)).iloc[0]
    return {k: row[k] for k in df.columns}


def pick_anchor(rng: random.Random, *, require_year: bool = False, require_member: bool = False, require_admin: bool = False) -> Optional[Dict[str, Any]]:
    df = get_anchor_pool()
    if df is None or len(df) == 0:
        return None
    sub = df.copy()
    if require_year:
        sub = sub[sub["inception_year"].str.fullmatch(r"\d{3,4}").fillna(False)]
    if require_member:
        sub = sub[sub["member_qid"].str.fullmatch(r"Q\d+").fillna(False)]
    if require_admin:
        sub = sub[sub["admin_qid"].str.fullmatch(r"Q\d+").fillna(False)]
    if len(sub) == 0:
        return None
    return sample_df_row(sub, rng)


def pick_membership(rng: random.Random, *, min_n: int = 3) -> Optional[Dict[str, Any]]:
    df = get_membership_pool()
    if df is None or len(df) == 0:
        return None
    sub = df.copy()
    if "n" in sub.columns:
        sub = sub[sub["n"].astype(str).astype(int) >= int(min_n)]
    return sample_df_row(sub, rng)


def pick_admin_area(rng: random.Random, *, min_n: int = 3) -> Optional[Dict[str, Any]]:
    df = get_admin_pool()
    if df is None or len(df) == 0:
        return None
    sub = df.copy()
    if "n" in sub.columns:
        sub = sub[sub["n"].astype(str).astype(int) >= int(min_n)]
    return sample_df_row(sub, rng)


def pick_country(rng: random.Random) -> Dict[str, str]:
    return rng.choice(COUNTRIES)

print("✅ lazy Wikidata pools ready")


✅ lazy Wikidata pools ready


In [17]:
# ============================================================
# ============================================================
# 7. Higher-quality relation helpers for L3-L5


In [18]:
# ============================================================

def static_same_country_lines_no_exclude(anchor: Dict[str, Any]) -> List[str]:
    return [
        f"wd:{anchor['qid']} wdt:P17 ?country .",
        "?item wdt:P17 ?country .",
    ]


def static_same_admin_lines_no_exclude(anchor: Dict[str, Any]) -> List[str]:
    return [
        f"wd:{anchor['qid']} wdt:P131 ?admin_area .",
        "?item wdt:P131 ?admin_area .",
    ]


def strict_between_anchor_years_lines(anchor_a: Dict[str, Any], anchor_b: Dict[str, Any]) -> Tuple[List[str], int, int]:
    low, high = sorted([int(anchor_a["year"]), int(anchor_b["year"])])
    # Strict interval avoids explicit "excluding" criteria while still preventing
    # the two anchors from becoming answers merely because their years define the range.
    return year_filter_lines(min_year=low + 1, max_year=high - 1), low + 1, high - 1


def nobel_alumni_lines(person_var: str = "nobel_person") -> List[str]:
    return [
        f"?{person_var} wdt:P69 ?item .",
        f"{{ ?{person_var} wdt:P166 wd:{Q_NOBEL_PRIZE} . }} UNION {{ ?{person_var} wdt:P166 ?nobel_award . ?nobel_award wdt:P279* wd:{Q_NOBEL_PRIZE} . }} UNION {{ ?{person_var} wdt:P166 ?nobel_award2 . ?nobel_award2 wdt:P31/wdt:P279* wd:{Q_NOBEL_PRIZE} . }}",
    ]


def nobel_alumni_born_in_country_lines(country_qid: str, person_var: str = "nobel_person") -> List[str]:
    return [
        *nobel_alumni_lines(person_var=person_var),
        f"?{person_var} wdt:P19 ?birth_place .",
        f"?birth_place wdt:P17 wd:{country_qid} .",
    ]


def membership_org_lines(org_qid: str) -> List[str]:
    return [f"?item wdt:P463 wd:{org_qid} ."]


def same_membership_as_anchor_lines_no_exclude(anchor: Dict[str, Any]) -> List[str]:
    return [
        f"wd:{anchor['qid']} wdt:P463 ?university_association .",
        "?item wdt:P463 ?university_association .",
    ]


def pick_membership_or_reject(rng: random.Random, *, min_n: int = 3) -> Optional[Dict[str, Any]]:
    org = pick_membership(rng, min_n=min_n)
    if not org:
        reject("membership_pool_empty")
        return None
    org_qid = qid_from_any(org.get("member_qid"))
    org_en = en_name(org.get("member_label_en"), org.get("member_label_ru"))
    org_ru = ru_name(org.get("member_label_ru"), org.get("member_label_en"))
    if not org_qid or not org_en or not org_ru:
        reject("bad_membership_pool_row")
        return None
    return {"qid": org_qid, "en": org_en, "ru": org_ru, "n": int(org.get("n") or 0)}


def pick_admin_anchor_or_reject(rng: random.Random) -> Optional[Dict[str, Any]]:
    anchor = pick_anchor(rng, require_year=True, require_admin=True)
    if not anchor:
        reject("admin_anchor_pool_empty")
        return None
    item_qid = qid_from_any(anchor.get("item_qid"))
    admin_qid = qid_from_any(anchor.get("admin_qid"))
    if not item_qid or not admin_qid:
        reject("bad_admin_anchor_pool_row")
        return None
    return {
        "qid": item_qid,
        "en": en_name(anchor.get("item_label_en"), anchor.get("item_label_ru")),
        "ru": ru_name(anchor.get("item_label_ru"), anchor.get("item_label_en")),
        "admin_qid": admin_qid,
        "admin_en": en_name(anchor.get("admin_label_en"), anchor.get("admin_label_ru")),
        "admin_ru": ru_name(anchor.get("admin_label_ru"), anchor.get("admin_label_en")),
        "year": int(anchor.get("inception_year") or 0),
    }

print("✅ v8 relation helpers ready: Nobel alumni, membership and admin-area patterns")


✅ v8 relation helpers ready: Nobel alumni, membership and admin-area patterns


In [19]:
# ============================================================
# 8. L1 templates: disabled by default


In [20]:
# ============================================================
# Country-only L1 records are intentionally not generated in v8. They were too
# easy and produced low-value benchmark examples. Keep an empty list so the
# schema and generation loop still support L1 if the target is manually changed.
L1_TEMPLATES: List[Callable[[int, random.Random], Optional[BenchmarkExample]]] = []
print(f"✅ L1 templates: {len(L1_TEMPLATES)} (disabled; target={TARGET_PER_LEVEL['L1']})")


✅ L1 templates: 0 (disabled; target=0)


In [21]:
# ============================================================
# 9. L2 templates: natural two-criterion filters


In [22]:
# ============================================================

def make_l2_country_inception_window(idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    country = pick_country(rng)
    y1, y2 = rng.choice(YEAR_WINDOWS_L2)
    k = REQUESTED_BY_LEVEL["L2"]
    where = [f"?item wdt:P17 wd:{country['qid']} .", *inception_window_lines(y1, y2)]
    return make_example(
        level="L2", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} страны «{country['ru']}», основанных в период {y1}–{y2}.",
        query_text_en=f"Name {k} universities in {country['en']} that were founded between {y1} and {y2}.",
        constraints={"kind": "university", "country": country["en"], "inception_year_from": y1, "inception_year_to": y2},
        where_lines=where,
        template_id="universities_l2_country_inception_window",
        template_family="country_year",
        is_advanced=False,
        constraint_entity_qids={"country": country["qid"]},
    )


def make_l2_country_before_year(idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    country = pick_country(rng)
    year = rng.choice(REFERENCE_YEARS)
    k = REQUESTED_BY_LEVEL["L2"]
    where = [f"?item wdt:P17 wd:{country['qid']} .", *before_year_lines(year)]
    return make_example(
        level="L2", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} страны «{country['ru']}», основанных раньше {year} года.",
        query_text_en=f"Name {k} universities in {country['en']} that were founded before {year}.",
        constraints={"kind": "university", "country": country["en"], "inception_year_before": year},
        where_lines=where,
        template_id="universities_l2_country_before_year",
        template_family="country_year",
        is_advanced=False,
        constraint_entity_qids={"country": country["qid"]},
    )


def make_l2_country_nobel_alumni(idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    country = pick_country(rng)
    k = REQUESTED_BY_LEVEL["L2"]
    where = [f"?item wdt:P17 wd:{country['qid']} .", *nobel_alumni_lines()]
    return make_example(
        level="L2", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} страны «{country['ru']}», где учился хотя бы один лауреат Нобелевской премии.",
        query_text_en=f"Name {k} universities in {country['en']} where at least one Nobel Prize laureate studied.",
        constraints={"kind": "university", "country": country["en"], "alumnus_award": "Nobel Prize"},
        where_lines=where,
        template_id="universities_l2_country_nobel_alumni",
        template_family="country_nobel_alumni",
        is_advanced=False,
        constraint_entity_qids={"country": country["qid"], "award": Q_NOBEL_PRIZE},
    )

L2_TEMPLATES = [make_l2_country_inception_window, make_l2_country_before_year, make_l2_country_nobel_alumni]
print(f"✅ L2 templates: {len(L2_TEMPLATES)}")


✅ L2 templates: 3


In [23]:
# ============================================================
# 10. L3 templates: real multihop, no country-only records


In [24]:
# ============================================================
# v9 note: Nobel Prize is now only one possible criterion. Most active
# L3-L5 templates use alumni occupation, staff award, founder occupation,
# named-after person occupation, or non-Nobel awards. Active L4/L5 templates
# avoid dynamic WDQS-built membership/admin pools, which made v8 extremely slow.

FAST_COUNTRY_QIDS = {"Q30", "Q145", "Q183", "Q142", "Q17", "Q39", "Q40", "Q55", "Q36", "Q258", "Q96", "Q408"}
FAST_COUNTRIES: List[Dict[str, str]] = [c for c in COUNTRIES if c["qid"] in FAST_COUNTRY_QIDS]
if not FAST_COUNTRIES:
    FAST_COUNTRIES = COUNTRIES

PEOPLE_OCCUPATIONS: List[Dict[str, str]] = [
    {"qid": "Q901", "en": "scientist", "ru": "учёный", "ru_pl": "учёные", "ru_inst": "учёным", "ru_gen": "учёного"},
    {"qid": "Q82955", "en": "politician", "ru": "политик", "ru_pl": "политики", "ru_inst": "политиком", "ru_gen": "политика"},
    {"qid": "Q36180", "en": "writer", "ru": "писатель", "ru_pl": "писатели", "ru_inst": "писателем", "ru_gen": "писателя"},
    {"qid": "Q188094", "en": "economist", "ru": "экономист", "ru_pl": "экономисты", "ru_inst": "экономистом", "ru_gen": "экономиста"},
    {"qid": "Q169470", "en": "physicist", "ru": "физик", "ru_pl": "физики", "ru_inst": "физиком", "ru_gen": "физика"},
    {"qid": "Q170790", "en": "mathematician", "ru": "математик", "ru_pl": "математики", "ru_inst": "математиком", "ru_gen": "математика"},
    {"qid": "Q81096", "en": "engineer", "ru": "инженер", "ru_pl": "инженеры", "ru_inst": "инженером", "ru_gen": "инженера"},
]

AWARD_CRITERIA: List[Dict[str, Any]] = [
    {"qid": "Q7191", "extra_qids": ["Q38104", "Q44585", "Q37922", "Q35637", "Q80061", "Q47170"], "en": "Nobel Prize", "ru": "Нобелевская премия"},
    {"qid": "Q28835", "extra_qids": [], "en": "Fields Medal", "ru": "Филдсовская премия"},
    {"qid": "Q185667", "extra_qids": [], "en": "Turing Award", "ru": "премия Тьюринга"},
]


def pick_fast_country(rng: random.Random) -> Dict[str, str]:
    return rng.choice(FAST_COUNTRIES)


def pick_occupation(rng: random.Random) -> Dict[str, str]:
    return rng.choice(PEOPLE_OCCUPATIONS)


def pick_award(rng: random.Random) -> Dict[str, Any]:
    return rng.choice(AWARD_CRITERIA)


def award_values_clause(award: Dict[str, Any], var: str = "award") -> str:
    qids = [award["qid"], *award.get("extra_qids", [])]
    values = " ".join(f"wd:{qid}" for qid in qids if re.fullmatch(r"Q\d+", str(qid)))
    return f"VALUES ?{var} {{ {values} }}"


def alumnus_occupation_lines(occupation_qid: str, person_var: str = "alumnus") -> List[str]:
    return [
        f"?{person_var} wdt:P69 ?item .",
        f"?{person_var} wdt:P106/wdt:P279* wd:{occupation_qid} .",
    ]


def alumnus_award_lines(award: Dict[str, Any], person_var: str = "award_alumnus") -> List[str]:
    return [
        f"?{person_var} wdt:P69 ?item .",
        f"?{person_var} wdt:P166 ?award .",
        award_values_clause(award, "award"),
    ]


def staff_award_lines(award: Dict[str, Any], person_var: str = "staff_person") -> List[str]:
    return [
        f"?{person_var} wdt:P108 ?item .",
        f"?{person_var} wdt:P166 ?staff_award .",
        award_values_clause(award, "staff_award"),
    ]


def staff_occupation_lines(occupation_qid: str, person_var: str = "staff_person") -> List[str]:
    return [
        f"?{person_var} wdt:P108 ?item .",
        f"?{person_var} wdt:P106/wdt:P279* wd:{occupation_qid} .",
    ]


def founder_occupation_lines(occupation_qid: str, person_var: str = "founder") -> List[str]:
    return [
        f"?item wdt:P112 ?{person_var} .",
        f"?{person_var} wdt:P106/wdt:P279* wd:{occupation_qid} .",
    ]


def named_after_occupation_lines(occupation_qid: str, person_var: str = "named_after") -> List[str]:
    return [
        f"?item wdt:P138 ?{person_var} .",
        f"?{person_var} wdt:P106/wdt:P279* wd:{occupation_qid} .",
    ]


def award_meta(award: Dict[str, Any]) -> Any:
    qids = [award["qid"], *award.get("extra_qids", [])]
    return qids if len(qids) > 1 else qids[0]


def make_l3_country_alumnus_occupation_window(idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    country = pick_fast_country(rng)
    occ = pick_occupation(rng)
    y1, y2 = rng.choice(YEAR_WINDOWS_L3)
    k = REQUESTED_BY_LEVEL["L3"]
    where = [f"?item wdt:P17 wd:{country['qid']} .", *alumnus_occupation_lines(occ["qid"]), *inception_window_lines(y1, y2)]
    return make_example(
        level="L3", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} страны «{country['ru']}», основанных в период {y1}–{y2}, среди выпускников которых есть {occ['ru_pl']}.",
        query_text_en=f"Name {k} universities in {country['en']} founded between {y1} and {y2} whose alumni include {occ['en']}s.",
        constraints={"kind": "university", "country": country["en"], "inception_year_from": y1, "inception_year_to": y2, "alumnus_occupation": occ["en"]},
        where_lines=where,
        template_id="universities_l3_country_alumnus_occupation_window",
        template_family="country_year_alumnus_occupation",
        is_advanced=True,
        constraint_entity_qids={"country": country["qid"], "alumnus_occupation": occ["qid"]},
    )


def make_l3_capital_country_alumnus_award(idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    country = pick_capital_country(rng)
    award = pick_award(rng)
    k = REQUESTED_BY_LEVEL["L3"]
    where = [*capital_country_lines(country), *alumnus_award_lines(award)]
    return make_example(
        level="L3", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} в стране, столицей которой является «{country['capital_ru']}», среди выпускников которых есть лауреаты награды «{award['ru']}».",
        query_text_en=f"Name {k} universities in the country whose capital is {country['capital_en']} whose alumni include winners of the {award['en']}.",
        constraints={"kind": "university", "country_capital": country["capital_en"], "alumnus_award": award["en"]},
        where_lines=where,
        template_id="universities_l3_capital_country_alumnus_award",
        template_family="country_capital_alumnus_award",
        is_advanced=True,
        constraint_entity_qids={"country": country["country_qid"], "capital": country["capital_qid"], "alumnus_award": award_meta(award)},
        derived_labels={"country": country["country_en"]},
    )


def make_l3_same_country_staff_occupation_after_anchor(idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    anchor = pick_static_anchor(rng)
    occ = pick_occupation(rng)
    k = REQUESTED_BY_LEVEL["L3"]
    where = [*static_same_country_lines_no_exclude(anchor), *staff_occupation_lines(occ["qid"]), *after_year_lines(anchor["year"])]
    return make_example(
        level="L3", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} из той же страны, что и «{anchor['ru']}», основанных позже этого университета, где работали {occ['ru_pl']}.",
        query_text_en=f"Name {k} universities from the same country as {anchor['en']} that were founded later than that university and employed {occ['en']}s.",
        constraints={"kind": "university", "country_from_university": anchor["en"], "founded_later_than_university": anchor["en"], "staff_occupation": occ["en"]},
        where_lines=where,
        template_id="universities_l3_same_country_staff_occupation_after_anchor",
        template_family="anchor_country_year_staff_occupation",
        is_advanced=True,
        constraint_entity_qids={"country_anchor_university": anchor["qid"], "year_anchor_university": anchor["qid"], "staff_occupation": occ["qid"], "country": anchor["country_qid"]},
        derived_labels={"country": anchor["country_en"], "anchor_year": anchor["year"]},
    )


def make_l3_country_founder_occupation(idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    country = pick_fast_country(rng)
    occ = rng.choice([o for o in PEOPLE_OCCUPATIONS if o["qid"] in {"Q82955", "Q901", "Q36180", "Q81096"}])
    k = REQUESTED_BY_LEVEL["L3"]
    where = [f"?item wdt:P17 wd:{country['qid']} .", *founder_occupation_lines(occ["qid"])]
    return make_example(
        level="L3", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} страны «{country['ru']}», основанных человеком, который был {occ['ru_inst']}.",
        query_text_en=f"Name {k} universities in {country['en']} that were founded by a person who was a {occ['en']}.",
        constraints={"kind": "university", "country": country["en"], "founder_occupation": occ["en"]},
        where_lines=where,
        template_id="universities_l3_country_founder_occupation",
        template_family="country_founder_occupation",
        is_advanced=True,
        constraint_entity_qids={"country": country["qid"], "founder_occupation": occ["qid"]},
    )


L3_TEMPLATES = [
    make_l3_country_alumnus_occupation_window,
    make_l3_capital_country_alumnus_award,
    make_l3_same_country_staff_occupation_after_anchor,
    make_l3_country_founder_occupation,
]
print(f"✅ L3 templates v9: {len(L3_TEMPLATES)}")


✅ L3 templates v9: 4


In [25]:
# ============================================================
# 11. L4 templates: faster diverse multi-criteria multihops


In [26]:
# ============================================================

def make_l4_country_alumnus_occupation_staff_award(idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    country = pick_fast_country(rng)
    occ = pick_occupation(rng)
    award = pick_award(rng)
    k = REQUESTED_BY_LEVEL["L4"]
    where = [f"?item wdt:P17 wd:{country['qid']} .", *alumnus_occupation_lines(occ["qid"]), *staff_award_lines(award)]
    return make_example(
        level="L4", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} страны «{country['ru']}», среди выпускников которых есть {occ['ru_pl']}, а среди сотрудников — лауреат награды «{award['ru']}».",
        query_text_en=f"Name {k} universities in {country['en']} whose alumni include {occ['en']}s and whose staff included a winner of the {award['en']}.",
        constraints={"kind": "university", "country": country["en"], "alumnus_occupation": occ["en"], "staff_award": award["en"]},
        where_lines=where,
        template_id="universities_l4_country_alumnus_occupation_staff_award",
        template_family="country_alumnus_occupation_staff_award",
        is_advanced=True,
        constraint_entity_qids={"country": country["qid"], "alumnus_occupation": occ["qid"], "staff_award": award_meta(award)},
    )


def make_l4_capital_country_window_alumnus_occupation(idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    country = pick_capital_country(rng)
    occ = pick_occupation(rng)
    y1, y2 = rng.choice(YEAR_WINDOWS_L4)
    k = REQUESTED_BY_LEVEL["L4"]
    where = [*capital_country_lines(country), *inception_window_lines(y1, y2), *alumnus_occupation_lines(occ["qid"])]
    return make_example(
        level="L4", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} в стране, столицей которой является «{country['capital_ru']}», основанных в период {y1}–{y2}, среди выпускников которых есть {occ['ru_pl']}.",
        query_text_en=f"Name {k} universities in the country whose capital is {country['capital_en']}, founded between {y1} and {y2}, whose alumni include {occ['en']}s.",
        constraints={"kind": "university", "country_capital": country["capital_en"], "inception_year_from": y1, "inception_year_to": y2, "alumnus_occupation": occ["en"]},
        where_lines=where,
        template_id="universities_l4_capital_country_window_alumnus_occupation",
        template_family="country_capital_year_alumnus_occupation",
        is_advanced=True,
        constraint_entity_qids={"country": country["country_qid"], "capital": country["capital_qid"], "alumnus_occupation": occ["qid"]},
        derived_labels={"country": country["country_en"]},
    )


def make_l4_same_country_after_anchor_alumnus_award(idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    anchor = pick_static_anchor(rng)
    award = pick_award(rng)
    k = REQUESTED_BY_LEVEL["L4"]
    where = [*static_same_country_lines_no_exclude(anchor), *after_year_lines(anchor["year"]), *alumnus_award_lines(award)]
    return make_example(
        level="L4", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} из той же страны, что и «{anchor['ru']}», основанных позже этого университета, среди выпускников которых есть лауреаты награды «{award['ru']}».",
        query_text_en=f"Name {k} universities from the same country as {anchor['en']} that were founded later than that university and whose alumni include winners of the {award['en']}.",
        constraints={"kind": "university", "country_from_university": anchor["en"], "founded_later_than_university": anchor["en"], "alumnus_award": award["en"]},
        where_lines=where,
        template_id="universities_l4_same_country_after_anchor_alumnus_award",
        template_family="anchor_country_year_alumnus_award",
        is_advanced=True,
        constraint_entity_qids={"country_anchor_university": anchor["qid"], "year_anchor_university": anchor["qid"], "country": anchor["country_qid"], "alumnus_award": award_meta(award)},
        derived_labels={"country": anchor["country_en"], "anchor_year": anchor["year"]},
    )


def make_l4_country_founder_occupation_alumnus_occupation(idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    country = pick_fast_country(rng)
    founder_occ = rng.choice([o for o in PEOPLE_OCCUPATIONS if o["qid"] in {"Q82955", "Q901", "Q36180", "Q81096"}])
    alumnus_occ = pick_occupation(rng)
    k = REQUESTED_BY_LEVEL["L4"]
    where = [f"?item wdt:P17 wd:{country['qid']} .", *founder_occupation_lines(founder_occ["qid"]), *alumnus_occupation_lines(alumnus_occ["qid"])]
    return make_example(
        level="L4", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} страны «{country['ru']}», основанных человеком, который был {founder_occ['ru_inst']}, и среди выпускников которых есть {alumnus_occ['ru_pl']}.",
        query_text_en=f"Name {k} universities in {country['en']} that were founded by a person who was a {founder_occ['en']} and whose alumni include {alumnus_occ['en']}s.",
        constraints={"kind": "university", "country": country["en"], "founder_occupation": founder_occ["en"], "alumnus_occupation": alumnus_occ["en"]},
        where_lines=where,
        template_id="universities_l4_country_founder_alumnus_occupation",
        template_family="country_founder_occupation_alumnus_occupation",
        is_advanced=True,
        constraint_entity_qids={"country": country["qid"], "founder_occupation": founder_occ["qid"], "alumnus_occupation": alumnus_occ["qid"]},
    )


def make_l4_country_named_after_occupation_window(idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    country = pick_fast_country(rng)
    occ = rng.choice([o for o in PEOPLE_OCCUPATIONS if o["qid"] in {"Q901", "Q82955", "Q36180", "Q169470"}])
    y1, y2 = rng.choice(YEAR_WINDOWS_L4)
    k = REQUESTED_BY_LEVEL["L4"]
    where = [f"?item wdt:P17 wd:{country['qid']} .", *named_after_occupation_lines(occ["qid"]), *inception_window_lines(y1, y2)]
    return make_example(
        level="L4", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} страны «{country['ru']}», названных в честь человека, который был {occ['ru_inst']}, и основанных в период {y1}–{y2}.",
        query_text_en=f"Name {k} universities in {country['en']} that are named after a person who was a {occ['en']} and were founded between {y1} and {y2}.",
        constraints={"kind": "university", "country": country["en"], "named_after_occupation": occ["en"], "inception_year_from": y1, "inception_year_to": y2},
        where_lines=where,
        template_id="universities_l4_country_named_after_occupation_window",
        template_family="country_named_after_occupation_year",
        is_advanced=True,
        constraint_entity_qids={"country": country["qid"], "named_after_occupation": occ["qid"]},
    )


L4_TEMPLATES = [
    make_l4_country_alumnus_occupation_staff_award,
    make_l4_capital_country_window_alumnus_occupation,
    make_l4_same_country_after_anchor_alumnus_award,
    make_l4_country_founder_occupation_alumnus_occupation,
    make_l4_country_named_after_occupation_window,
]
print(f"✅ L4 templates v9: {len(L4_TEMPLATES)} (no active dynamic pools)")


✅ L4 templates v9: 5 (no active dynamic pools)


In [27]:
# ============================================================
# 12. L5 templates: hard multihop and multi-criterion, but not Nobel-only


In [28]:
# ============================================================

def make_l5_anchor_country_between_years_alumnus_staff(idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    anchor_a, anchor_b = pick_static_anchor_pair(rng)
    lines, y1, y2 = strict_between_anchor_years_lines(anchor_a, anchor_b)
    if y1 > y2:
        return None
    alumnus_occ = pick_occupation(rng)
    staff_occ = pick_occupation(rng)
    if staff_occ["qid"] == alumnus_occ["qid"]:
        staff_occ = rng.choice([o for o in PEOPLE_OCCUPATIONS if o["qid"] != alumnus_occ["qid"]])
    k = REQUESTED_BY_LEVEL["L5"]
    where = [*static_same_country_lines_no_exclude(anchor_a), *lines, *alumnus_occupation_lines(alumnus_occ["qid"]), *staff_occupation_lines(staff_occ["qid"])]
    return make_example(
        level="L5", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} из той же страны, что и «{anchor_a['ru']}», основанных после года основания «{anchor_a['ru']}» и до года основания «{anchor_b['ru']}», среди выпускников которых есть {alumnus_occ['ru_pl']}, а среди сотрудников — {staff_occ['ru_pl']}.",
        query_text_en=f"Name {k} universities from the same country as {anchor_a['en']} that were founded after the founding year of {anchor_a['en']} and before the founding year of {anchor_b['en']}, whose alumni include {alumnus_occ['en']}s and whose staff included {staff_occ['en']}s.",
        constraints={"kind": "university", "country_from_university": anchor_a["en"], "founded_after_university": anchor_a["en"], "founded_before_university": anchor_b["en"], "alumnus_occupation": alumnus_occ["en"], "staff_occupation": staff_occ["en"]},
        where_lines=where,
        template_id="universities_l5_anchor_country_between_years_alumnus_staff",
        template_family="two_anchor_country_year_alumnus_staff",
        is_advanced=True,
        constraint_entity_qids={"country_anchor_university": anchor_a["qid"], "after_year_university": anchor_a["qid"], "before_year_university": anchor_b["qid"], "country": anchor_a["country_qid"], "alumnus_occupation": alumnus_occ["qid"], "staff_occupation": staff_occ["qid"]},
        derived_labels={"country": anchor_a["country_en"], "year_from": y1, "year_to": y2},
    )


def make_l5_capital_window_alumnus_award_staff_occupation(idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    country = pick_capital_country(rng)
    y1, y2 = rng.choice(YEAR_WINDOWS_L5)
    award = pick_award(rng)
    staff_occ = pick_occupation(rng)
    k = REQUESTED_BY_LEVEL["L5"]
    where = [*capital_country_lines(country), *inception_window_lines(y1, y2), *alumnus_award_lines(award), *staff_occupation_lines(staff_occ["qid"])]
    return make_example(
        level="L5", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} в стране, столицей которой является «{country['capital_ru']}», основанных в период {y1}–{y2}, среди выпускников которых есть лауреаты награды «{award['ru']}», а среди сотрудников — {staff_occ['ru_pl']}.",
        query_text_en=f"Name {k} universities in the country whose capital is {country['capital_en']}, founded between {y1} and {y2}, whose alumni include winners of the {award['en']} and whose staff included {staff_occ['en']}s.",
        constraints={"kind": "university", "country_capital": country["capital_en"], "inception_year_from": y1, "inception_year_to": y2, "alumnus_award": award["en"], "staff_occupation": staff_occ["en"]},
        where_lines=where,
        template_id="universities_l5_capital_window_alumnus_award_staff_occupation",
        template_family="country_capital_year_alumnus_award_staff_occupation",
        is_advanced=True,
        constraint_entity_qids={"country": country["country_qid"], "capital": country["capital_qid"], "alumnus_award": award_meta(award), "staff_occupation": staff_occ["qid"]},
        derived_labels={"country": country["country_en"]},
    )


def make_l5_country_year_alumnus_award_founder_occupation(idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    country = pick_fast_country(rng)
    year = rng.choice([1800, 1850, 1900, 1950])
    award = pick_award(rng)
    founder_occ = rng.choice([o for o in PEOPLE_OCCUPATIONS if o["qid"] in {"Q82955", "Q901", "Q36180", "Q81096"}])
    k = REQUESTED_BY_LEVEL["L5"]
    where = [f"?item wdt:P17 wd:{country['qid']} .", *after_year_lines(year), *alumnus_award_lines(award), *founder_occupation_lines(founder_occ["qid"])]
    return make_example(
        level="L5", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} страны «{country['ru']}», основанных позже {year} года человеком, который был {founder_occ['ru_inst']}, и среди выпускников которых есть лауреаты награды «{award['ru']}».",
        query_text_en=f"Name {k} universities in {country['en']} that were founded after {year} by a person who was a {founder_occ['en']} and whose alumni include winners of the {award['en']}.",
        constraints={"kind": "university", "country": country["en"], "inception_year_after": year, "founder_occupation": founder_occ["en"], "alumnus_award": award["en"]},
        where_lines=where,
        template_id="universities_l5_country_year_alumnus_award_founder_occupation",
        template_family="country_year_founder_occupation_alumnus_award",
        is_advanced=True,
        constraint_entity_qids={"country": country["qid"], "founder_occupation": founder_occ["qid"], "alumnus_award": award_meta(award)},
    )


def make_l5_country_alumnus_award_alumnus_occupation_staff_award(idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    country = pick_fast_country(rng)
    award_alumnus = pick_award(rng)
    award_staff = pick_award(rng)
    occ = pick_occupation(rng)
    k = REQUESTED_BY_LEVEL["L5"]
    where = [f"?item wdt:P17 wd:{country['qid']} .", *alumnus_award_lines(award_alumnus), *alumnus_occupation_lines(occ["qid"], person_var="occupation_alumnus"), *staff_award_lines(award_staff)]
    return make_example(
        level="L5", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} страны «{country['ru']}», среди выпускников которых есть {occ['ru_pl']} и лауреаты награды «{award_alumnus['ru']}», а среди сотрудников — лауреат награды «{award_staff['ru']}».",
        query_text_en=f"Name {k} universities in {country['en']} whose alumni include {occ['en']}s and winners of the {award_alumnus['en']}, and whose staff included a winner of the {award_staff['en']}.",
        constraints={"kind": "university", "country": country["en"], "alumnus_occupation": occ["en"], "alumnus_award": award_alumnus["en"], "staff_award": award_staff["en"]},
        where_lines=where,
        template_id="universities_l5_country_alumnus_award_occupation_staff_award",
        template_family="country_alumnus_award_occupation_staff_award",
        is_advanced=True,
        constraint_entity_qids={"country": country["qid"], "alumnus_occupation": occ["qid"], "alumnus_award": award_meta(award_alumnus), "staff_award": award_meta(award_staff)},
    )


def make_l5_country_named_after_founder_alumnus_occupation(idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    country = pick_fast_country(rng)
    named_occ = rng.choice([o for o in PEOPLE_OCCUPATIONS if o["qid"] in {"Q901", "Q82955", "Q36180", "Q169470"}])
    founder_occ = rng.choice([o for o in PEOPLE_OCCUPATIONS if o["qid"] in {"Q82955", "Q901", "Q36180", "Q81096"}])
    alumnus_occ = pick_occupation(rng)
    k = REQUESTED_BY_LEVEL["L5"]
    where = [f"?item wdt:P17 wd:{country['qid']} .", *named_after_occupation_lines(named_occ["qid"]), *founder_occupation_lines(founder_occ["qid"]), *alumnus_occupation_lines(alumnus_occ["qid"])]
    return make_example(
        level="L5", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} страны «{country['ru']}», названных в честь {named_occ['ru_gen']}, основанных {founder_occ['ru_inst']}, и среди выпускников которых есть {alumnus_occ['ru_pl']}.",
        query_text_en=f"Name {k} universities in {country['en']} that are named after a {named_occ['en']}, were founded by a {founder_occ['en']}, and whose alumni include {alumnus_occ['en']}s.",
        constraints={"kind": "university", "country": country["en"], "named_after_occupation": named_occ["en"], "founder_occupation": founder_occ["en"], "alumnus_occupation": alumnus_occ["en"]},
        where_lines=where,
        template_id="universities_l5_country_named_after_founder_alumnus_occupation",
        template_family="country_named_after_founder_alumnus_occupation",
        is_advanced=True,
        constraint_entity_qids={"country": country["qid"], "named_after_occupation": named_occ["qid"], "founder_occupation": founder_occ["qid"], "alumnus_occupation": alumnus_occ["qid"]},
    )


L5_TEMPLATES = [
    make_l5_anchor_country_between_years_alumnus_staff,
    make_l5_capital_window_alumnus_award_staff_occupation,
    make_l5_country_year_alumnus_award_founder_occupation,
    make_l5_country_alumnus_award_alumnus_occupation_staff_award,
    make_l5_country_named_after_founder_alumnus_occupation,
]
print(f"✅ L5 templates v9: {len(L5_TEMPLATES)} (diverse non-dynamic patterns)")


✅ L5 templates v9: 5 (diverse non-dynamic patterns)


In [29]:
# ============================================================
# 12. Generation loop


In [30]:
# ============================================================
TEMPLATES_BY_LEVEL: Dict[str, List[Callable[[int, random.Random], Optional[BenchmarkExample]]]] = {
    "L1": L1_TEMPLATES,
    "L2": L2_TEMPLATES,
    "L3": L3_TEMPLATES,
    "L4": L4_TEMPLATES,
    "L5": L5_TEMPLATES,
}


def semantic_key(ex: BenchmarkExample) -> Tuple[Any, ...]:
    return (ex.domain, ex.complexity, ex.template_id, json.dumps(ex.constraints, ensure_ascii=False, sort_keys=True))


def gold_key(ex: BenchmarkExample) -> Tuple[str, ...]:
    return tuple(sorted(ex.gold_answer_qids))


def validate_example_basic(ex: BenchmarkExample) -> List[str]:
    errors: List[str] = []
    obj = ordered_as_benchmark_example(ex)
    if list(obj.keys()) != EXPECTED_KEYS:
        errors.append("schema_key_order_mismatch")
    if ex.domain != DOMAIN:
        errors.append("wrong_domain")
    if ex.complexity not in LEVELS:
        errors.append("wrong_complexity")
    if not question_text_ok(ex.query_text_ru) or not question_text_ok(ex.query_text_en):
        errors.append("bad_query_text")
    if len(ex.gold_answer_qids) == 0:
        errors.append("zero_gold")
    if len(ex.gold_answer_qids) < ex.requested_count:
        errors.append("gold_less_than_requested")
    if len(ex.gold_answer_qids) != len(ex.gold_answer_labels_ru):
        errors.append("ru_gold_label_length_mismatch")
    if len(ex.gold_answer_qids) != len(ex.gold_answer_labels_en):
        errors.append("en_gold_label_length_mismatch")
    if ex.gold_truncated:
        errors.append("gold_truncated_true")
    if "wd:{ITEM}" not in (ex.ask_validator_sparql or ""):
        errors.append("ask_validator_missing_placeholder")
    if not constraints_are_clean(ex.constraints):
        errors.append("bad_constraints")

    semantic_criteria = [k for k in (ex.constraints or {}) if k != "kind"]
    if ex.complexity in {"L4", "L5"} and len(semantic_criteria) < 3:
        errors.append("too_few_semantic_constraints_for_level")
    if any(QID_ONLY_RE.fullmatch(str(x or "")) for x in ex.gold_answer_labels_ru):
        errors.append("qid_as_ru_label")
    if any(QID_ONLY_RE.fullmatch(str(x or "")) for x in ex.gold_answer_labels_en):
        errors.append("qid_as_en_label")
    return errors


def generate_one(level: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    templates = TEMPLATES_BY_LEVEL[level]
    template = rng.choice(templates)
    return template(idx, rng)


def generate_universities_dataset(
    *,
    out_path: Path = OUTPUT_PATH,
    target_per_level: Dict[str, int] = TARGET_PER_LEVEL,
    seed: int = SEED,
    max_attempts_per_level: int = MAX_ATTEMPTS_PER_LEVEL,
    overwrite: bool = OVERWRITE_OUTPUT,
) -> List[BenchmarkExample]:
    rng = random.Random(seed)
    if overwrite and out_path.exists():
        out_path.unlink()

    examples: List[BenchmarkExample] = []
    seen_semantic = set()
    seen_gold = set()
    reject_counts: Counter = Counter()

    for level in LEVELS:
        target = int(target_per_level.get(level, 0))
        accepted = 0
        attempts = 0
        while accepted < target and attempts < max_attempts_per_level:
            attempts += 1
            idx = accepted + 1
            global LAST_REJECT_REASON
            LAST_REJECT_REASON = "none"
            try:
                ex = generate_one(level, idx, rng)
            except Exception as e:
                reject_counts[(level, "exception")] += 1
                if DEBUG_GENERATOR_ERRORS:
                    print(f"[WARN] {level} attempt {attempts}: {e}")
                continue
            if ex is None:
                reject_counts[(level, LAST_REJECT_REASON or "none")] += 1
                continue
            errors = validate_example_basic(ex)
            if errors:
                reject_counts[(level, ",".join(errors))] += 1
                continue
            sk = semantic_key(ex)
            gk = gold_key(ex)
            if sk in seen_semantic:
                reject_counts[(level, "duplicate_semantic")] += 1
                continue
            if gk in seen_gold:
                reject_counts[(level, "duplicate_gold_set")] += 1
                continue
            seen_semantic.add(sk)
            seen_gold.add(gk)
            examples.append(ex)
            accepted += 1
            append_example_jsonl(out_path, ex)
            print(f"[{DOMAIN}] {level}: {accepted}/{target} saved — {ex.template_id} — gold={len(ex.gold_answer_qids)}")
        if accepted < target:
            print(f"[WARN] {DOMAIN}:{level} generated {accepted}/{target} after {attempts} attempts")
            print("[WARN] common reject reasons:", reject_counts.most_common(12))
    print(f"✅ generation finished: {len(examples)} examples saved to {out_path}")
    return examples

print("✅ generation loop ready")


✅ generation loop ready


In [31]:
# ============================================================
# 13. Final JSONL validation


In [32]:
# ============================================================

def validate_output_file(path: Path = OUTPUT_PATH) -> Dict[str, Any]:
    rows = read_jsonl(path)
    report: Dict[str, Any] = {
        "path": str(path),
        "total": len(rows),
        "by_level": dict(Counter(row.get("complexity") for row in rows)),
        "errors": [],
        "warnings": [],
        "template_counts": dict(Counter(row.get("template_id") for row in rows)),
    }
    ids = set()
    semantic_seen = set()
    gold_seen = set()

    for i, row in enumerate(rows, 1):
        prefix = f"line_{i}:{row.get('id', '<no id>')}"
        if list(row.keys()) != EXPECTED_KEYS:
            report["errors"].append(f"{prefix}:schema_key_order_mismatch")
        rid = row.get("id")
        if rid in ids:
            report["errors"].append(f"{prefix}:duplicate_id")
        ids.add(rid)
        if row.get("domain") != DOMAIN:
            report["errors"].append(f"{prefix}:wrong_domain")
        if not question_text_ok(row.get("query_text_ru", "")) or not question_text_ok(row.get("query_text_en", "")):
            report["errors"].append(f"{prefix}:bad_query_text")
        constraints = row.get("constraints") or {}
        if not constraints_are_clean(constraints):
            report["errors"].append(f"{prefix}:bad_constraints")
        semantic_criteria = [k for k in constraints if k != "kind"]
        if row.get("complexity") in {"L4", "L5"} and len(semantic_criteria) < 3:
            report["errors"].append(f"{prefix}:too_few_semantic_constraints_for_level")
        requested = int(row.get("requested_count") or 0)
        qids = row.get("gold_answer_qids") or []
        ru_labels = row.get("gold_answer_labels_ru") or []
        en_labels = row.get("gold_answer_labels_en") or []
        if not qids:
            report["errors"].append(f"{prefix}:zero_gold")
        if len(qids) < requested:
            report["errors"].append(f"{prefix}:gold_less_than_requested")
        if len(qids) != len(ru_labels):
            report["errors"].append(f"{prefix}:ru_label_length_mismatch")
        if len(qids) != len(en_labels):
            report["errors"].append(f"{prefix}:en_label_length_mismatch")
        if row.get("gold_truncated"):
            report["errors"].append(f"{prefix}:gold_truncated_true")
        if "wd:{ITEM}" not in str(row.get("ask_validator_sparql") or ""):
            report["errors"].append(f"{prefix}:ask_validator_missing_placeholder")
        if any(QID_ONLY_RE.fullmatch(str(x or "")) for x in ru_labels):
            report["errors"].append(f"{prefix}:qid_as_ru_label")
        if any(QID_ONLY_RE.fullmatch(str(x or "")) for x in en_labels):
            report["errors"].append(f"{prefix}:qid_as_en_label")
        meta = row.get("gold_collection_meta") or {}
        if not isinstance(meta, dict) or not meta.get("constraint_entity_qids"):
            report["warnings"].append(f"{prefix}:missing_constraint_entity_qids_meta")
        sk = (row.get("domain"), row.get("complexity"), row.get("template_id"), json.dumps(constraints, ensure_ascii=False, sort_keys=True))
        if sk in semantic_seen:
            report["warnings"].append(f"{prefix}:duplicate_semantic_key")
        semantic_seen.add(sk)
        gk = tuple(sorted(qids))
        if gk in gold_seen:
            report["warnings"].append(f"{prefix}:duplicate_gold_set")
        gold_seen.add(gk)

    for level, target in TARGET_PER_LEVEL.items():
        actual = int(report["by_level"].get(level, 0) or 0)
        if actual != int(target):
            report["errors"].append(f"level_count_mismatch:{level}:{actual}!={target}")

    report_path = path.with_suffix(".validation_report.json")
    with report_path.open("w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)
    print(json.dumps(report, ensure_ascii=False, indent=2)[:5000])
    print(f"✅ validation report saved to {report_path}")
    return report


def show_sample(path: Path = OUTPUT_PATH, n: int = 5) -> None:
    rows = read_jsonl(path)
    for row in rows[:n]:
        print("=" * 80)
        print(row.get("id"), row.get("complexity"), row.get("template_id"))
        print(row.get("query_text_ru"))
        print(row.get("query_text_en"))
        print("constraints:", json.dumps(row.get("constraints"), ensure_ascii=False))
        print("meta.constraint_entity_qids:", (row.get("gold_collection_meta") or {}).get("constraint_entity_qids"))
        print("gold:", len(row.get("gold_answer_qids") or []), row.get("gold_answer_labels_en", [])[:5])

print("✅ validation helpers ready")


✅ validation helpers ready


In [33]:

# ============================================================
# 13b. v10 overrides: quality gold, faster deterministic candidates, stronger L4/L5
# ============================================================
# This cell intentionally overrides several v9 functions/templates before the
# final run cell executes. It keeps the top-level BenchmarkExample schema intact,
# but fixes the v9 blockers found in generated files:
#   * noisy gold entities (faculties, schools, centres, campuses, colleges, institutes);
#   * L2 dominated by country+year;
#   * broad staff_occupation=scientist queries;
#   * slow random L3+ attempts and missing L5;
#   * final validation that did not fail on dirty gold labels.

VERSION = "v11"
MAX_ATTEMPTS_PER_LEVEL = 4500
MAX_CANDIDATES_PER_LEVEL = {"L1": 0, "L2": 160, "L3": 220, "L4": 260, "L5": 340}
MAX_TEMPLATE_SHARE = {"L2": 0.45, "L3": 0.35, "L4": 0.35, "L5": 0.30}

# Fetch fewer broad rows and reject huge noisy answers earlier. Since the SPARQL
# itself now filters bad labels before LIMIT, these caps are caps on quality gold.
MAX_GOLD_BY_LEVEL.update({"L2": 90, "L3": 85, "L4": 70, "L5": 60})

# SPARQL-safe broad blacklist. Python post-filter below is stricter and catches
# additional institutional-unit labels. The SPARQL part reduces transfer size and
# makes truncation meaningful after obvious bad labels are removed.
SPARQL_BAD_LABEL_PATTERN = (
    "faculty|department|school of|medical school|business school|graduate school|"
    "centre|center|seminary|campus|training centre|training center|student union|"
    "athletic union|winter school|summer school|research network|security network|"
    "school for|college of|unit[ae]?d?\\s+xochimilco|unit[ae]?d?\\s+azcapotzalco|unit[ae]?d?\\s+iztapalapa"
)

_BAD_LABEL_RE = re.compile(
    r"(?i)("
    r"\bfaculty\b|\bdepartment\b|\bschool of\b|\bmedical school\b|\bbusiness school\b|"
    r"\bgraduate school\b|\bseminary\b|\bcampus\b|\bcentre\b|\bcenter\b|"
    r"\btraining centre\b|\btraining center\b|\bathletic union\b|\bstudent union\b|"
    r"\bwinter school\b|\bsummer school\b|\bresearch network\b|\bsecurity network\b|"
    r"\binstitute\b|\bacademy\b|\bcollege\b|\bunit(?:ad|à)?\b|"
    r"la casa de los famosos|wellcome institute|university centre|school for|"
    r"school,|college,|polytekniske læreanstalt|higher school|normal school|teachers.? college|"
    r"athrolys|university of science, arts and technology|university of life|carolinum|benedict schools|benedict international education group|russian and eurasian security network|higher school of latvia|normal university"
    r")"
)
_GOOD_LABEL_EXCEPTIONS_RE = re.compile(
    r"(?i)("
    r"Massachusetts Institute of Technology|California Institute of Technology|"
    r"Illinois Institute of Technology|Stevens Institute of Technology|"
    r"Georgia Institute of Technology|Tokyo Institute of Technology|"
    r"Indian Institute of Technology|Karlsruhe Institute of Technology|"
    r"National Polytechnic Institute|Polytechnic University|Technical University|"
    r"Technology University|Institute of Technology|Technological University|"
    r"University College London|University College Dublin"
    r")"
)


def university_label_quality_ok(label_en: Any, label_ru: Any = "") -> bool:
    label = clean_string(label_en) or clean_string(label_ru)
    joined = clean_string(f"{label_en} {label_ru}")
    if not label_ok(label):
        return False
    if _GOOD_LABEL_EXCEPTIONS_RE.search(joined):
        return True
    if _BAD_LABEL_RE.search(joined):
        return False
    return True


def label_block(item_var: str = "item") -> str:
    # Overrides v9 label_block. The quality filter is inside SPARQL so LIMIT is
    # applied after obvious institutional-unit labels are removed.
    return f'''OPTIONAL {{ ?{item_var} rdfs:label ?{item_var}LabelRu FILTER(LANG(?{item_var}LabelRu) = "ru") . }}
      OPTIONAL {{ ?{item_var} rdfs:label ?{item_var}LabelEn FILTER(LANG(?{item_var}LabelEn) = "en") . }}
      FILTER(BOUND(?{item_var}LabelRu) || BOUND(?{item_var}LabelEn)) .
      BIND(LCASE(CONCAT(COALESCE(STR(?{item_var}LabelEn), ""), " ", COALESCE(STR(?{item_var}LabelRu), ""))) AS ?{item_var}QualityLabel) .
      FILTER(!REGEX(?{item_var}QualityLabel, "{SPARQL_BAD_LABEL_PATTERN}", "i")) .'''


def quality_ask_lines(item_var: str = "item") -> List[str]:
    return [
        f'OPTIONAL {{ ?{item_var} rdfs:label ?{item_var}LabelRu FILTER(LANG(?{item_var}LabelRu) = "ru") . }}',
        f'OPTIONAL {{ ?{item_var} rdfs:label ?{item_var}LabelEn FILTER(LANG(?{item_var}LabelEn) = "en") . }}',
        f'FILTER(BOUND(?{item_var}LabelRu) || BOUND(?{item_var}LabelEn)) .',
        f'BIND(LCASE(CONCAT(COALESCE(STR(?{item_var}LabelEn), ""), " ", COALESCE(STR(?{item_var}LabelRu), ""))) AS ?{item_var}QualityLabel) .',
        f'FILTER(!REGEX(?{item_var}QualityLabel, "{SPARQL_BAD_LABEL_PATTERN}", "i")) .',
    ]


def build_ask_sparql(where_lines: Sequence[str], *, item_var: str = "item", direct_only: bool = DIRECT_INSTANCE_ONLY) -> str:
    # Overrides v9 build_ask_sparql so ASK validators reject obvious non-university
    # institutional units in the same way as SELECT gold collection.
    where = "\n      ".join(dedupe_where_lines([*where_lines, *quality_ask_lines(item_var)]))
    return f'''
    ASK WHERE {{
      BIND(wd:{{ITEM}} AS ?{item_var})
      {class_membership_line(item_var=item_var, direct_only=direct_only)}
      {where}
    }}
    '''.strip()


def rows_to_gold(rows: List[Dict[str, str]], *, item_var: str = "item") -> Tuple[List[str], List[str], List[str]]:
    # Overrides v9 rows_to_gold with a strict post-filter. This is intentionally
    # redundant with SPARQL filtering; it catches labels that slip through because
    # of language variants or names such as "... College" / "... Institute".
    qids: List[str] = []
    labels_ru: List[str] = []
    labels_en: List[str] = []
    seen = set()
    for row in rows:
        qid = qid_from_any(row.get(item_var))
        if not qid or qid in seen:
            continue
        label_ru = first_good_label(row.get(f"{item_var}LabelRu"), row.get(f"{item_var}LabelEn"))
        label_en = first_good_label(row.get(f"{item_var}LabelEn"), row.get(f"{item_var}LabelRu"))
        if not label_ok(label_ru) or not label_ok(label_en):
            continue
        if not university_label_quality_ok(label_en, label_ru):
            continue
        seen.add(qid)
        qids.append(qid)
        labels_ru.append(label_ru)
        labels_en.append(label_en)
    return qids, labels_ru, labels_en


def validate_gold_label_quality(labels_en: Sequence[str], labels_ru: Sequence[str]) -> List[str]:
    bad = []
    for en, ru in zip(labels_en or [], labels_ru or []):
        if not university_label_quality_ok(en, ru):
            bad.append(en or ru)
    return bad


def validate_example_basic(ex: BenchmarkExample) -> List[str]:
    # Overrides v9 validator; now dirty gold labels are fatal, not just a manual review note.
    errors: List[str] = []
    obj = ordered_as_benchmark_example(ex)
    if list(obj.keys()) != EXPECTED_KEYS:
        errors.append("schema_key_order_mismatch")
    if ex.domain != DOMAIN:
        errors.append("wrong_domain")
    if ex.complexity not in LEVELS:
        errors.append("wrong_complexity")
    if not question_text_ok(ex.query_text_ru) or not question_text_ok(ex.query_text_en):
        errors.append("bad_query_text")
    if len(ex.gold_answer_qids) == 0:
        errors.append("zero_gold")
    if len(ex.gold_answer_qids) < ex.requested_count:
        errors.append("gold_less_than_requested")
    if len(ex.gold_answer_qids) != len(ex.gold_answer_labels_ru):
        errors.append("ru_gold_label_length_mismatch")
    if len(ex.gold_answer_qids) != len(ex.gold_answer_labels_en):
        errors.append("en_gold_label_length_mismatch")
    if ex.gold_truncated:
        errors.append("gold_truncated_true")
    if "wd:{ITEM}" not in (ex.ask_validator_sparql or ""):
        errors.append("ask_validator_missing_placeholder")
    if not constraints_are_clean(ex.constraints):
        errors.append("bad_constraints")
    bad_labels = validate_gold_label_quality(ex.gold_answer_labels_en, ex.gold_answer_labels_ru)
    if bad_labels:
        errors.append("bad_gold_labels:" + ";".join(bad_labels[:5]))
    semantic_criteria = [k for k in (ex.constraints or {}) if k != "kind"]
    if ex.complexity == "L3" and len(semantic_criteria) < 2:
        errors.append("too_few_semantic_constraints_for_L3")
    if ex.complexity in {"L4", "L5"} and len(semantic_criteria) < 3:
        errors.append("too_few_semantic_constraints_for_level")
    if ex.complexity == "L5" and len(semantic_criteria) < 4:
        errors.append("too_few_semantic_constraints_for_L5")
    return errors

# Curated pools for speed and precision. Broad "scientist" is not used as a random
# occupation anymore; it appears only in tightly constrained legacy-style founder
# templates where it is not a broad staff/alumni expansion.
RICH_COUNTRY_QIDS = {"Q30", "Q145", "Q17", "Q258", "Q36", "Q40", "Q55", "Q35", "Q33", "Q183", "Q142", "Q96"}
RICH_COUNTRIES = [c for c in COUNTRIES if c["qid"] in RICH_COUNTRY_QIDS]
if not RICH_COUNTRIES:
    RICH_COUNTRIES = FAST_COUNTRIES

STRICT_OCCUPATIONS = [
    {"qid": "Q82955", "en": "politician", "ru": "политик", "ru_pl": "политики", "ru_inst": "политиком", "ru_gen": "политика"},
    {"qid": "Q36180", "en": "writer", "ru": "писатель", "ru_pl": "писатели", "ru_inst": "писателем", "ru_gen": "писателя"},
    {"qid": "Q188094", "en": "economist", "ru": "экономист", "ru_pl": "экономисты", "ru_inst": "экономистом", "ru_gen": "экономиста"},
    {"qid": "Q169470", "en": "physicist", "ru": "физик", "ru_pl": "физики", "ru_inst": "физиком", "ru_gen": "физика"},
    {"qid": "Q170790", "en": "mathematician", "ru": "математик", "ru_pl": "математики", "ru_inst": "математиком", "ru_gen": "математика"},
    {"qid": "Q81096", "en": "engineer", "ru": "инженер", "ru_pl": "инженеры", "ru_inst": "инженером", "ru_gen": "инженера"},
    {"qid": "Q82594", "en": "computer scientist", "ru": "специалист по информатике", "ru_pl": "специалисты по информатике", "ru_inst": "специалистом по информатике", "ru_gen": "специалиста по информатике"},
]
FOUNDER_OCCUPATIONS = [
    {"qid": "Q901", "en": "scientist", "ru": "учёный", "ru_pl": "учёные", "ru_inst": "учёным", "ru_gen": "учёного"},
    *[o for o in STRICT_OCCUPATIONS if o["qid"] in {"Q82955", "Q36180", "Q81096", "Q169470"}],
]
NAMED_AFTER_OCCUPATIONS = [o for o in STRICT_OCCUPATIONS if o["qid"] in {"Q82955", "Q36180", "Q169470", "Q170790"}]


def pick_rich_country(rng: random.Random) -> Dict[str, str]:
    return rng.choice(RICH_COUNTRIES)


def pick_occupation(rng: random.Random) -> Dict[str, str]:
    return rng.choice(STRICT_OCCUPATIONS)


def country_by_qid(qid: str) -> Dict[str, str]:
    return next(c for c in COUNTRIES if c["qid"] == qid)


def award_by_en(name: str) -> Dict[str, Any]:
    return next(a for a in AWARD_CRITERIA if a["en"] == name)


def occ_by_en(name: str, pool: Sequence[Dict[str, str]] = STRICT_OCCUPATIONS) -> Dict[str, str]:
    return next(o for o in pool if o["en"] == name)

# -------------------------
# v10 deterministic builders
# -------------------------

def make_l2_country_alumnus_award_v10(idx: int, rng: random.Random, *, country: Optional[Dict[str, str]] = None, award: Optional[Dict[str, Any]] = None) -> Optional[BenchmarkExample]:
    country = country or pick_rich_country(rng)
    award = award or rng.choice(AWARD_CRITERIA)
    k = REQUESTED_BY_LEVEL["L2"]
    where = [f"?item wdt:P17 wd:{country['qid']} .", *alumnus_award_lines(award)]
    return make_example(
        level="L2", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} страны «{country['ru']}», среди выпускников которых есть лауреаты награды «{award['ru']}».",
        query_text_en=f"Name {k} universities in {country['en']} whose alumni include winners of the {award['en']}.",
        constraints={"kind": "university", "country": country["en"], "alumnus_award": award["en"]},
        where_lines=where,
        template_id="universities_l2_country_alumnus_award_v10",
        template_family="country_alumnus_award",
        is_advanced=False,
        constraint_entity_qids={"country": country["qid"], "alumnus_award": award_meta(award)},
    )


def make_l2_country_founder_occupation_v10(idx: int, rng: random.Random, *, country: Optional[Dict[str, str]] = None, occ: Optional[Dict[str, str]] = None) -> Optional[BenchmarkExample]:
    country = country or pick_rich_country(rng)
    occ = occ or rng.choice(FOUNDER_OCCUPATIONS)
    k = REQUESTED_BY_LEVEL["L2"]
    where = [f"?item wdt:P17 wd:{country['qid']} .", *founder_occupation_lines(occ["qid"])]
    return make_example(
        level="L2", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} страны «{country['ru']}», основанных человеком, который был {occ['ru_inst']}.",
        query_text_en=f"Name {k} universities in {country['en']} that were founded by a person who was a {occ['en']}.",
        constraints={"kind": "university", "country": country["en"], "founder_occupation": occ["en"]},
        where_lines=where,
        template_id="universities_l2_country_founder_occupation_v10",
        template_family="country_founder_occupation",
        is_advanced=False,
        constraint_entity_qids={"country": country["qid"], "founder_occupation": occ["qid"]},
    )


def make_l2_country_staff_award_v10(idx: int, rng: random.Random, *, country: Optional[Dict[str, str]] = None, award: Optional[Dict[str, Any]] = None) -> Optional[BenchmarkExample]:
    country = country or pick_rich_country(rng)
    award = award or rng.choice(AWARD_CRITERIA)
    k = REQUESTED_BY_LEVEL["L2"]
    where = [f"?item wdt:P17 wd:{country['qid']} .", *staff_award_lines(award)]
    return make_example(
        level="L2", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} страны «{country['ru']}», где среди сотрудников был лауреат награды «{award['ru']}».",
        query_text_en=f"Name {k} universities in {country['en']} whose staff included a winner of the {award['en']}.",
        constraints={"kind": "university", "country": country["en"], "staff_award": award["en"]},
        where_lines=where,
        template_id="universities_l2_country_staff_award_v10",
        template_family="country_staff_award",
        is_advanced=False,
        constraint_entity_qids={"country": country["qid"], "staff_award": award_meta(award)},
    )


def make_l3_country_year_alumnus_occupation_v10(idx: int, rng: random.Random, *, country: Optional[Dict[str, str]] = None, occ: Optional[Dict[str, str]] = None, window: Optional[Tuple[int, int]] = None) -> Optional[BenchmarkExample]:
    country = country or pick_rich_country(rng)
    occ = occ or pick_occupation(rng)
    y1, y2 = window or rng.choice(YEAR_WINDOWS_L3)
    k = REQUESTED_BY_LEVEL["L3"]
    where = [f"?item wdt:P17 wd:{country['qid']} .", *inception_window_lines(y1, y2), *alumnus_occupation_lines(occ["qid"])]
    return make_example(
        level="L3", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} страны «{country['ru']}», основанных в период {y1}–{y2}, среди выпускников которых есть {occ['ru_pl']}.",
        query_text_en=f"Name {k} universities in {country['en']} founded between {y1} and {y2} whose alumni include {occ['en']}s.",
        constraints={"kind": "university", "country": country["en"], "inception_year_from": y1, "inception_year_to": y2, "alumnus_occupation": occ["en"]},
        where_lines=where,
        template_id="universities_l3_country_year_alumnus_occupation_v10",
        template_family="country_year_alumnus_occupation",
        is_advanced=True,
        constraint_entity_qids={"country": country["qid"], "alumnus_occupation": occ["qid"]},
    )


def make_l3_country_founder_alumnus_occupation_v10(idx: int, rng: random.Random, *, country: Optional[Dict[str, str]] = None, founder_occ: Optional[Dict[str, str]] = None, alumnus_occ: Optional[Dict[str, str]] = None) -> Optional[BenchmarkExample]:
    country = country or pick_rich_country(rng)
    founder_occ = founder_occ or rng.choice(FOUNDER_OCCUPATIONS)
    alumnus_occ = alumnus_occ or pick_occupation(rng)
    if founder_occ["qid"] == alumnus_occ["qid"]:
        alumnus_occ = occ_by_en("physicist") if alumnus_occ["en"] != "physicist" else occ_by_en("economist")
    k = REQUESTED_BY_LEVEL["L3"]
    where = [f"?item wdt:P17 wd:{country['qid']} .", *founder_occupation_lines(founder_occ["qid"]), *alumnus_occupation_lines(alumnus_occ["qid"])]
    return make_example(
        level="L3", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} страны «{country['ru']}», основанных человеком, который был {founder_occ['ru_inst']}, и среди выпускников которых есть {alumnus_occ['ru_pl']}.",
        query_text_en=f"Name {k} universities in {country['en']} founded by a person who was a {founder_occ['en']} and whose alumni include {alumnus_occ['en']}s.",
        constraints={"kind": "university", "country": country["en"], "founder_occupation": founder_occ["en"], "alumnus_occupation": alumnus_occ["en"]},
        where_lines=where,
        template_id="universities_l3_country_founder_alumnus_occupation_v10",
        template_family="country_founder_alumnus_occupation",
        is_advanced=True,
        constraint_entity_qids={"country": country["qid"], "founder_occupation": founder_occ["qid"], "alumnus_occupation": alumnus_occ["qid"]},
    )


def make_l3_capital_year_alumnus_award_v10(idx: int, rng: random.Random, *, country: Optional[Dict[str, Any]] = None, award: Optional[Dict[str, Any]] = None, window: Optional[Tuple[int, int]] = None) -> Optional[BenchmarkExample]:
    country = country or pick_capital_country(rng)
    award = award or rng.choice(AWARD_CRITERIA)
    y1, y2 = window or rng.choice(YEAR_WINDOWS_L3)
    k = REQUESTED_BY_LEVEL["L3"]
    where = [*capital_country_lines(country), *inception_window_lines(y1, y2), *alumnus_award_lines(award)]
    return make_example(
        level="L3", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} в стране, столицей которой является «{country['capital_ru']}», основанных в период {y1}–{y2}, среди выпускников которых есть лауреаты награды «{award['ru']}».",
        query_text_en=f"Name {k} universities in the country whose capital is {country['capital_en']}, founded between {y1} and {y2}, whose alumni include winners of the {award['en']}.",
        constraints={"kind": "university", "country_capital": country["capital_en"], "inception_year_from": y1, "inception_year_to": y2, "alumnus_award": award["en"]},
        where_lines=where,
        template_id="universities_l3_capital_year_alumnus_award_v10",
        template_family="country_capital_year_alumnus_award",
        is_advanced=True,
        constraint_entity_qids={"country": country["country_qid"], "capital": country["capital_qid"], "alumnus_award": award_meta(award)},
        derived_labels={"country": country["country_en"]},
    )


def make_l4_country_founder_alumnus_award_v10(idx: int, rng: random.Random, *, country: Optional[Dict[str, str]] = None, founder_occ: Optional[Dict[str, str]] = None, award: Optional[Dict[str, Any]] = None) -> Optional[BenchmarkExample]:
    country = country or pick_rich_country(rng)
    founder_occ = founder_occ or rng.choice(FOUNDER_OCCUPATIONS)
    award = award or rng.choice(AWARD_CRITERIA)
    k = REQUESTED_BY_LEVEL["L4"]
    where = [f"?item wdt:P17 wd:{country['qid']} .", *founder_occupation_lines(founder_occ["qid"]), *alumnus_award_lines(award)]
    return make_example(
        level="L4", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} страны «{country['ru']}», основанных человеком, который был {founder_occ['ru_inst']}, и среди выпускников которых есть лауреаты награды «{award['ru']}».",
        query_text_en=f"Name {k} universities in {country['en']} that were founded by a person who was a {founder_occ['en']} and whose alumni include winners of the {award['en']}.",
        constraints={"kind": "university", "country": country["en"], "founder_occupation": founder_occ["en"], "alumnus_award": award["en"]},
        where_lines=where,
        template_id="universities_l4_country_founder_alumnus_award_v10",
        template_family="country_founder_alumnus_award",
        is_advanced=True,
        constraint_entity_qids={"country": country["qid"], "founder_occupation": founder_occ["qid"], "alumnus_award": award_meta(award)},
    )


def make_l4_country_alumnus_occupation_staff_award_v10(idx: int, rng: random.Random, *, country: Optional[Dict[str, str]] = None, occ: Optional[Dict[str, str]] = None, award: Optional[Dict[str, Any]] = None) -> Optional[BenchmarkExample]:
    country = country or pick_rich_country(rng)
    occ = occ or pick_occupation(rng)
    award = award or rng.choice(AWARD_CRITERIA)
    k = REQUESTED_BY_LEVEL["L4"]
    where = [f"?item wdt:P17 wd:{country['qid']} .", *alumnus_occupation_lines(occ["qid"]), *staff_award_lines(award)]
    return make_example(
        level="L4", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} страны «{country['ru']}», среди выпускников которых есть {occ['ru_pl']}, а среди сотрудников был лауреат награды «{award['ru']}».",
        query_text_en=f"Name {k} universities in {country['en']} whose alumni include {occ['en']}s and whose staff included a winner of the {award['en']}.",
        constraints={"kind": "university", "country": country["en"], "alumnus_occupation": occ["en"], "staff_award": award["en"]},
        where_lines=where,
        template_id="universities_l4_country_alumnus_occupation_staff_award_v10",
        template_family="country_alumnus_occupation_staff_award",
        is_advanced=True,
        constraint_entity_qids={"country": country["qid"], "alumnus_occupation": occ["qid"], "staff_award": award_meta(award)},
    )


def make_l4_capital_year_alumnus_occupation_v10(idx: int, rng: random.Random, *, country: Optional[Dict[str, Any]] = None, occ: Optional[Dict[str, str]] = None, window: Optional[Tuple[int, int]] = None) -> Optional[BenchmarkExample]:
    country = country or pick_capital_country(rng)
    occ = occ or pick_occupation(rng)
    y1, y2 = window or rng.choice(YEAR_WINDOWS_L4)
    k = REQUESTED_BY_LEVEL["L4"]
    where = [*capital_country_lines(country), *inception_window_lines(y1, y2), *alumnus_occupation_lines(occ["qid"])]
    return make_example(
        level="L4", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} в стране, столицей которой является «{country['capital_ru']}», основанных в период {y1}–{y2}, среди выпускников которых есть {occ['ru_pl']}.",
        query_text_en=f"Name {k} universities in the country whose capital is {country['capital_en']}, founded between {y1} and {y2}, whose alumni include {occ['en']}s.",
        constraints={"kind": "university", "country_capital": country["capital_en"], "inception_year_from": y1, "inception_year_to": y2, "alumnus_occupation": occ["en"]},
        where_lines=where,
        template_id="universities_l4_capital_year_alumnus_occupation_v10",
        template_family="country_capital_year_alumnus_occupation",
        is_advanced=True,
        constraint_entity_qids={"country": country["country_qid"], "capital": country["capital_qid"], "alumnus_occupation": occ["qid"]},
        derived_labels={"country": country["country_en"]},
    )


def make_l4_same_country_after_anchor_alumnus_award_v10(idx: int, rng: random.Random, *, anchor: Optional[Dict[str, Any]] = None, award: Optional[Dict[str, Any]] = None) -> Optional[BenchmarkExample]:
    anchor = anchor or pick_static_anchor(rng)
    award = award or rng.choice(AWARD_CRITERIA)
    k = REQUESTED_BY_LEVEL["L4"]
    where = [*static_same_country_lines_no_exclude(anchor), *after_year_lines(int(anchor["year"])), *alumnus_award_lines(award)]
    return make_example(
        level="L4", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} из той же страны, что и «{anchor['ru']}», основанных позже года основания этого университета, среди выпускников которых есть лауреаты награды «{award['ru']}».",
        query_text_en=f"Name {k} universities from the same country as {anchor['en']} that were founded after that university's founding year and whose alumni include winners of the {award['en']}.",
        constraints={"kind": "university", "country_from_university": anchor["en"], "founded_later_than_university": anchor["en"], "alumnus_award": award["en"]},
        where_lines=where,
        template_id="universities_l4_same_country_after_anchor_alumnus_award_v10",
        template_family="anchor_country_year_alumnus_award",
        is_advanced=True,
        constraint_entity_qids={"country_anchor_university": anchor["qid"], "year_anchor_university": anchor["qid"], "country": anchor["country_qid"], "alumnus_award": award_meta(award)},
        derived_labels={"country": anchor["country_en"], "anchor_year": anchor["year"]},
    )


def make_l5_country_founder_alumnus_award_occupation_v10(idx: int, rng: random.Random, *, country: Optional[Dict[str, str]] = None, founder_occ: Optional[Dict[str, str]] = None, award: Optional[Dict[str, Any]] = None, alumnus_occ: Optional[Dict[str, str]] = None) -> Optional[BenchmarkExample]:
    country = country or pick_rich_country(rng)
    founder_occ = founder_occ or rng.choice(FOUNDER_OCCUPATIONS)
    award = award or rng.choice(AWARD_CRITERIA)
    alumnus_occ = alumnus_occ or pick_occupation(rng)
    k = REQUESTED_BY_LEVEL["L5"]
    where = [f"?item wdt:P17 wd:{country['qid']} .", *founder_occupation_lines(founder_occ["qid"]), *alumnus_award_lines(award), *alumnus_occupation_lines(alumnus_occ["qid"], person_var="occupation_alumnus")]
    return make_example(
        level="L5", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} страны «{country['ru']}», основанных человеком, который был {founder_occ['ru_inst']}, среди выпускников которых есть {alumnus_occ['ru_pl']} и лауреаты награды «{award['ru']}».",
        query_text_en=f"Name {k} universities in {country['en']} that were founded by a person who was a {founder_occ['en']}, and whose alumni include {alumnus_occ['en']}s and winners of the {award['en']}.",
        constraints={"kind": "university", "country": country["en"], "founder_occupation": founder_occ["en"], "alumnus_occupation": alumnus_occ["en"], "alumnus_award": award["en"]},
        where_lines=where,
        template_id="universities_l5_country_founder_award_occupation_v10",
        template_family="country_founder_alumnus_award_occupation",
        is_advanced=True,
        constraint_entity_qids={"country": country["qid"], "founder_occupation": founder_occ["qid"], "alumnus_occupation": alumnus_occ["qid"], "alumnus_award": award_meta(award)},
    )


def make_l5_country_alumnus_staff_award_occupation_v10(idx: int, rng: random.Random, *, country: Optional[Dict[str, str]] = None, alumnus_award: Optional[Dict[str, Any]] = None, staff_award: Optional[Dict[str, Any]] = None, occ: Optional[Dict[str, str]] = None) -> Optional[BenchmarkExample]:
    country = country or pick_rich_country(rng)
    alumnus_award = alumnus_award or rng.choice(AWARD_CRITERIA)
    staff_award = staff_award or rng.choice(AWARD_CRITERIA)
    occ = occ or pick_occupation(rng)
    k = REQUESTED_BY_LEVEL["L5"]
    where = [f"?item wdt:P17 wd:{country['qid']} .", *alumnus_award_lines(alumnus_award), *alumnus_occupation_lines(occ["qid"], person_var="occupation_alumnus"), *staff_award_lines(staff_award)]
    return make_example(
        level="L5", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} страны «{country['ru']}», среди выпускников которых есть {occ['ru_pl']} и лауреаты награды «{alumnus_award['ru']}», а среди сотрудников был лауреат награды «{staff_award['ru']}».",
        query_text_en=f"Name {k} universities in {country['en']} whose alumni include {occ['en']}s and winners of the {alumnus_award['en']}, and whose staff included a winner of the {staff_award['en']}.",
        constraints={"kind": "university", "country": country["en"], "alumnus_occupation": occ["en"], "alumnus_award": alumnus_award["en"], "staff_award": staff_award["en"]},
        where_lines=where,
        template_id="universities_l5_country_alumnus_award_occupation_staff_award_v10",
        template_family="country_alumnus_award_occupation_staff_award",
        is_advanced=True,
        constraint_entity_qids={"country": country["qid"], "alumnus_occupation": occ["qid"], "alumnus_award": award_meta(alumnus_award), "staff_award": award_meta(staff_award)},
    )


def make_l5_capital_year_award_staff_occupation_v10(idx: int, rng: random.Random, *, country: Optional[Dict[str, Any]] = None, award: Optional[Dict[str, Any]] = None, staff_occ: Optional[Dict[str, str]] = None, window: Optional[Tuple[int, int]] = None) -> Optional[BenchmarkExample]:
    country = country or pick_capital_country(rng)
    award = award or rng.choice(AWARD_CRITERIA)
    staff_occ = staff_occ or pick_occupation(rng)
    y1, y2 = window or rng.choice(YEAR_WINDOWS_L5)
    k = REQUESTED_BY_LEVEL["L5"]
    where = [*capital_country_lines(country), *inception_window_lines(y1, y2), *alumnus_award_lines(award), *staff_occupation_lines(staff_occ["qid"])]
    return make_example(
        level="L5", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} в стране, столицей которой является «{country['capital_ru']}», основанных в период {y1}–{y2}, среди выпускников которых есть лауреаты награды «{award['ru']}», а среди сотрудников — {staff_occ['ru_pl']}.",
        query_text_en=f"Name {k} universities in the country whose capital is {country['capital_en']}, founded between {y1} and {y2}, whose alumni include winners of the {award['en']} and whose staff included {staff_occ['en']}s.",
        constraints={"kind": "university", "country_capital": country["capital_en"], "inception_year_from": y1, "inception_year_to": y2, "alumnus_award": award["en"], "staff_occupation": staff_occ["en"]},
        where_lines=where,
        template_id="universities_l5_capital_year_award_staff_occupation_v10",
        template_family="country_capital_year_alumnus_award_staff_occupation",
        is_advanced=True,
        constraint_entity_qids={"country": country["country_qid"], "capital": country["capital_qid"], "alumnus_award": award_meta(award), "staff_occupation": staff_occ["qid"]},
        derived_labels={"country": country["country_en"]},
    )


def make_l5_same_country_after_anchor_award_staff_award_v10(idx: int, rng: random.Random, *, anchor: Optional[Dict[str, Any]] = None, alumnus_award: Optional[Dict[str, Any]] = None, staff_award: Optional[Dict[str, Any]] = None) -> Optional[BenchmarkExample]:
    anchor = anchor or pick_static_anchor(rng)
    alumnus_award = alumnus_award or rng.choice(AWARD_CRITERIA)
    staff_award = staff_award or rng.choice(AWARD_CRITERIA)
    k = REQUESTED_BY_LEVEL["L5"]
    where = [*static_same_country_lines_no_exclude(anchor), *after_year_lines(int(anchor["year"])), *alumnus_award_lines(alumnus_award), *staff_award_lines(staff_award)]
    return make_example(
        level="L5", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} из той же страны, что и «{anchor['ru']}», основанных позже года основания этого университета, среди выпускников которых есть лауреаты награды «{alumnus_award['ru']}», а среди сотрудников был лауреат награды «{staff_award['ru']}».",
        query_text_en=f"Name {k} universities from the same country as {anchor['en']} that were founded after that university's founding year, whose alumni include winners of the {alumnus_award['en']}, and whose staff included a winner of the {staff_award['en']}.",
        constraints={"kind": "university", "country_from_university": anchor["en"], "founded_later_than_university": anchor["en"], "alumnus_award": alumnus_award["en"], "staff_award": staff_award["en"]},
        where_lines=where,
        template_id="universities_l5_same_country_after_anchor_award_staff_award_v10",
        template_family="anchor_country_year_alumnus_award_staff_award",
        is_advanced=True,
        constraint_entity_qids={"country_anchor_university": anchor["qid"], "year_anchor_university": anchor["qid"], "country": anchor["country_qid"], "alumnus_award": award_meta(alumnus_award), "staff_award": award_meta(staff_award)},
        derived_labels={"country": anchor["country_en"], "anchor_year": anchor["year"]},
    )

# Active v10 templates. Country+year-only templates are removed from default L2.
L2_TEMPLATES = [make_l2_country_alumnus_award_v10, make_l2_country_founder_occupation_v10, make_l2_country_staff_award_v10]
L3_TEMPLATES = [make_l3_country_year_alumnus_occupation_v10, make_l3_country_founder_alumnus_occupation_v10, make_l3_capital_year_alumnus_award_v10]
L4_TEMPLATES = [make_l4_country_founder_alumnus_award_v10, make_l4_country_alumnus_occupation_staff_award_v10, make_l4_capital_year_alumnus_occupation_v10, make_l4_same_country_after_anchor_alumnus_award_v10]
L5_TEMPLATES = [make_l5_country_founder_alumnus_award_occupation_v10, make_l5_country_alumnus_staff_award_occupation_v10, make_l5_capital_year_award_staff_occupation_v10, make_l5_same_country_after_anchor_award_staff_award_v10]
TEMPLATES_BY_LEVEL = {"L1": L1_TEMPLATES, "L2": L2_TEMPLATES, "L3": L3_TEMPLATES, "L4": L4_TEMPLATES, "L5": L5_TEMPLATES}

# -------------------------
# Deterministic candidate banks
# -------------------------

def _bind(builder: Callable[..., Optional[BenchmarkExample]], **kwargs: Any) -> Callable[[int, random.Random], Optional[BenchmarkExample]]:
    def _candidate(idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
        return builder(idx, rng, **kwargs)
    _candidate.__name__ = builder.__name__
    return _candidate


def build_candidate_bank(level: str, rng: random.Random) -> List[Callable[[int, random.Random], Optional[BenchmarkExample]]]:
    bank: List[Callable[[int, random.Random], Optional[BenchmarkExample]]] = []
    awards = AWARD_CRITERIA
    nobel = award_by_en("Nobel Prize")
    fields = award_by_en("Fields Medal")
    turing = award_by_en("Turing Award")
    rich = RICH_COUNTRIES
    occs = STRICT_OCCUPATIONS
    founders = FOUNDER_OCCUPATIONS
    capitals = [c for c in CAPITAL_COUNTRIES if c["country_qid"] in RICH_COUNTRY_QIDS]
    anchors = [a for a in STATIC_ANCHORS if a["country_qid"] in RICH_COUNTRY_QIDS]

    if level == "L2":
        for c in rich:
            for a in [nobel, fields, turing]:
                bank.append(_bind(make_l2_country_alumnus_award_v10, country=c, award=a))
                bank.append(_bind(make_l2_country_staff_award_v10, country=c, award=a))
            for o in founders:
                bank.append(_bind(make_l2_country_founder_occupation_v10, country=c, occ=o))

    elif level == "L3":
        for c in rich:
            for o in occs:
                for w in YEAR_WINDOWS_L3:
                    bank.append(_bind(make_l3_country_year_alumnus_occupation_v10, country=c, occ=o, window=w))
            for fo in founders:
                for ao in occs:
                    if fo["qid"] != ao["qid"]:
                        bank.append(_bind(make_l3_country_founder_alumnus_occupation_v10, country=c, founder_occ=fo, alumnus_occ=ao))
        for cc in capitals:
            for a in awards:
                for w in YEAR_WINDOWS_L3:
                    bank.append(_bind(make_l3_capital_year_alumnus_award_v10, country=cc, award=a, window=w))

    elif level == "L4":
        for c in rich:
            for fo in founders:
                for a in awards:
                    bank.append(_bind(make_l4_country_founder_alumnus_award_v10, country=c, founder_occ=fo, award=a))
            for ao in occs:
                for a in awards:
                    bank.append(_bind(make_l4_country_alumnus_occupation_staff_award_v10, country=c, occ=ao, award=a))
        for cc in capitals:
            for o in occs:
                for w in YEAR_WINDOWS_L4:
                    bank.append(_bind(make_l4_capital_year_alumnus_occupation_v10, country=cc, occ=o, window=w))
        for anchor in anchors:
            for a in awards:
                bank.append(_bind(make_l4_same_country_after_anchor_alumnus_award_v10, anchor=anchor, award=a))

    elif level == "L5":
        # Rich/high-yield combinations first: these tend to generate quickly and
        # still have 4 semantic criteria after label-quality filtering.
        preferred_countries = [country_by_qid(qid) for qid in ["Q30", "Q145", "Q17", "Q258", "Q36", "Q40", "Q55", "Q183", "Q142", "Q96"] if any(c["qid"] == qid for c in COUNTRIES)]
        for c in preferred_countries:
            for fo in founders:
                for a in awards:
                    for ao in occs:
                        bank.append(_bind(make_l5_country_founder_alumnus_award_occupation_v10, country=c, founder_occ=fo, award=a, alumnus_occ=ao))
            for aa in awards:
                for sa in awards:
                    for o in occs:
                        bank.append(_bind(make_l5_country_alumnus_staff_award_occupation_v10, country=c, alumnus_award=aa, staff_award=sa, occ=o))
        for cc in capitals:
            for a in awards:
                for so in occs:
                    for w in YEAR_WINDOWS_L5:
                        bank.append(_bind(make_l5_capital_year_award_staff_occupation_v10, country=cc, award=a, staff_occ=so, window=w))
        for anchor in anchors:
            for aa in awards:
                for sa in awards:
                    bank.append(_bind(make_l5_same_country_after_anchor_award_staff_award_v10, anchor=anchor, alumnus_award=aa, staff_award=sa))

    rng.shuffle(bank)
    limit = int(MAX_CANDIDATES_PER_LEVEL.get(level, len(bank)))
    return bank[:limit]


def generate_universities_dataset(
    *,
    out_path: Path = OUTPUT_PATH,
    target_per_level: Dict[str, int] = TARGET_PER_LEVEL,
    seed: int = SEED,
    max_attempts_per_level: int = MAX_ATTEMPTS_PER_LEVEL,
    overwrite: bool = OVERWRITE_OUTPUT,
) -> List[BenchmarkExample]:
    # Overrides v9 random loop. v10 tries a shuffled deterministic candidate bank
    # first, which avoids spending thousands of attempts on the same failed L4/L5
    # templates and makes L3+ much faster/reproducible.
    rng = random.Random(seed)
    if overwrite and out_path.exists():
        out_path.unlink()
    examples: List[BenchmarkExample] = []
    seen_semantic = set()
    seen_gold = set()
    reject_counts: Counter = Counter()
    template_counts_by_level: Dict[str, Counter] = {level: Counter() for level in LEVELS}

    for level in LEVELS:
        target = int(target_per_level.get(level, 0))
        if target <= 0:
            continue
        accepted = 0
        attempts = 0
        bank = build_candidate_bank(level, rng)
        print(f"[{DOMAIN}] {level}: candidate bank={len(bank)} target={target}")
        # Candidate bank first.
        for candidate in bank:
            if accepted >= target:
                break
            attempts += 1
            idx = accepted + 1
            global LAST_REJECT_REASON
            LAST_REJECT_REASON = "none"
            try:
                ex = candidate(idx, rng)
            except Exception as e:
                reject_counts[(level, "exception", str(e)[:120])] += 1
                if DEBUG_GENERATOR_ERRORS:
                    print(f"[WARN] {level} candidate {attempts}: {e}")
                continue
            if ex is None:
                reject_counts[(level, LAST_REJECT_REASON or "none")] += 1
                continue
            errors = validate_example_basic(ex)
            if errors:
                reject_counts[(level, ",".join(errors[:2]))] += 1
                continue
            share_limit = max(1, int(target * MAX_TEMPLATE_SHARE.get(level, 1.0)))
            if template_counts_by_level[level][ex.template_id] >= share_limit:
                reject_counts[(level, "template_quota", ex.template_id)] += 1
                continue
            sk = semantic_key(ex)
            gk = gold_key(ex)
            if sk in seen_semantic:
                reject_counts[(level, "duplicate_semantic")] += 1
                continue
            if gk in seen_gold:
                reject_counts[(level, "duplicate_gold_set")] += 1
                continue
            seen_semantic.add(sk)
            seen_gold.add(gk)
            template_counts_by_level[level][ex.template_id] += 1
            examples.append(ex)
            accepted += 1
            append_example_jsonl(out_path, ex)
            print(f"[{DOMAIN}] {level}: {accepted}/{target} saved — {ex.template_id} — gold={len(ex.gold_answer_qids)}")

        # Short randomized fallback over v10 templates only. This is intentionally
        # bounded; if it cannot finish, final validation fails instead of silently
        # producing an incomplete file.
        fallback_attempts = 0
        while accepted < target and fallback_attempts < max_attempts_per_level:
            fallback_attempts += 1
            attempts += 1
            idx = accepted + 1
            LAST_REJECT_REASON = "none"
            try:
                ex = generate_one(level, idx, rng)
            except Exception as e:
                reject_counts[(level, "fallback_exception", str(e)[:120])] += 1
                continue
            if ex is None:
                reject_counts[(level, LAST_REJECT_REASON or "none")] += 1
                continue
            errors = validate_example_basic(ex)
            if errors:
                reject_counts[(level, ",".join(errors[:2]))] += 1
                continue
            share_limit = max(1, int(target * MAX_TEMPLATE_SHARE.get(level, 1.0)))
            if template_counts_by_level[level][ex.template_id] >= share_limit:
                reject_counts[(level, "template_quota", ex.template_id)] += 1
                continue
            sk = semantic_key(ex)
            gk = gold_key(ex)
            if sk in seen_semantic or gk in seen_gold:
                reject_counts[(level, "duplicate")]+=1
                continue
            seen_semantic.add(sk)
            seen_gold.add(gk)
            template_counts_by_level[level][ex.template_id] += 1
            examples.append(ex)
            accepted += 1
            append_example_jsonl(out_path, ex)
            print(f"[{DOMAIN}] {level}: {accepted}/{target} saved — {ex.template_id} — gold={len(ex.gold_answer_qids)} [fallback]")

        if accepted < target:
            print(f"[WARN] {DOMAIN}:{level} generated {accepted}/{target} after {attempts} attempts")
            print("[WARN] common reject reasons:", reject_counts.most_common(15))
        else:
            print(f"✅ {DOMAIN}:{level} complete in {attempts} WDQS candidate attempts; templates={dict(template_counts_by_level[level])}")
    print(f"✅ generation finished: {len(examples)} examples saved to {out_path}")
    return examples


def validate_output_file(path: Path = OUTPUT_PATH) -> Dict[str, Any]:
    # Overrides v9 final validator with distribution, gold-label-quality, and
    # template-diversity checks.
    rows = read_jsonl(path)
    report: Dict[str, Any] = {
        "path": str(path),
        "generator_version": f"universities_{VERSION}",
        "total": len(rows),
        "target_per_level": dict(TARGET_PER_LEVEL),
        "by_level": dict(Counter(row.get("complexity") for row in rows)),
        "errors": [],
        "warnings": [],
        "template_counts": dict(Counter(row.get("template_id") for row in rows)),
        "bad_gold_label_examples": [],
    }
    ids = set(); semantic_seen = set(); gold_seen = set()
    template_by_level: Dict[str, Counter] = {level: Counter() for level in LEVELS}
    for i, row in enumerate(rows, 1):
        prefix = f"line_{i}:{row.get('id', '<no id>')}"
        if list(row.keys()) != EXPECTED_KEYS:
            report["errors"].append(f"{prefix}:schema_key_order_mismatch")
        rid = row.get("id")
        if rid in ids:
            report["errors"].append(f"{prefix}:duplicate_id")
        ids.add(rid)
        if row.get("domain") != DOMAIN:
            report["errors"].append(f"{prefix}:wrong_domain")
        if not question_text_ok(row.get("query_text_ru", "")) or not question_text_ok(row.get("query_text_en", "")):
            report["errors"].append(f"{prefix}:bad_query_text")
        constraints = row.get("constraints") or {}
        if not constraints_are_clean(constraints):
            report["errors"].append(f"{prefix}:bad_constraints")
        semantic_criteria = [k for k in constraints if k != "kind"]
        level = row.get("complexity")
        if level == "L3" and len(semantic_criteria) < 2:
            report["errors"].append(f"{prefix}:too_few_semantic_constraints_for_L3")
        if level in {"L4", "L5"} and len(semantic_criteria) < 3:
            report["errors"].append(f"{prefix}:too_few_semantic_constraints_for_level")
        if level == "L5" and len(semantic_criteria) < 4:
            report["errors"].append(f"{prefix}:too_few_semantic_constraints_for_L5")
        requested = int(row.get("requested_count") or 0)
        qids = row.get("gold_answer_qids") or []
        ru_labels = row.get("gold_answer_labels_ru") or []
        en_labels = row.get("gold_answer_labels_en") or []
        if not qids:
            report["errors"].append(f"{prefix}:zero_gold")
        if len(qids) < requested:
            report["errors"].append(f"{prefix}:gold_less_than_requested")
        if len(qids) != len(ru_labels):
            report["errors"].append(f"{prefix}:ru_label_length_mismatch")
        if len(qids) != len(en_labels):
            report["errors"].append(f"{prefix}:en_label_length_mismatch")
        if row.get("gold_truncated"):
            report["errors"].append(f"{prefix}:gold_truncated_true")
        if "wd:{ITEM}" not in str(row.get("ask_validator_sparql") or ""):
            report["errors"].append(f"{prefix}:ask_validator_missing_placeholder")
        bad_labels = validate_gold_label_quality(en_labels, ru_labels)
        if bad_labels:
            report["errors"].append(f"{prefix}:bad_gold_labels")
            report["bad_gold_label_examples"].append({"id": row.get("id"), "labels": bad_labels[:10]})
        meta = row.get("gold_collection_meta") or {}
        if not isinstance(meta, dict) or not meta.get("constraint_entity_qids"):
            report["warnings"].append(f"{prefix}:missing_constraint_entity_qids_meta")
        if level in template_by_level:
            template_by_level[level][row.get("template_id")] += 1
        sk = (row.get("domain"), row.get("complexity"), row.get("template_id"), json.dumps(constraints, ensure_ascii=False, sort_keys=True))
        if sk in semantic_seen:
            report["warnings"].append(f"{prefix}:duplicate_semantic_key")
        semantic_seen.add(sk)
        gk = tuple(sorted(qids))
        if gk in gold_seen:
            report["warnings"].append(f"{prefix}:duplicate_gold_set")
        gold_seen.add(gk)

    for level, target in TARGET_PER_LEVEL.items():
        actual = int(report["by_level"].get(level, 0) or 0)
        if actual != int(target):
            report["errors"].append(f"level_count_mismatch:{level}:{actual}!={target}")
    for level, counts in template_by_level.items():
        target = int(TARGET_PER_LEVEL.get(level, 0) or 0)
        if target <= 0 or not counts:
            continue
        share_limit = max(1, int(target * MAX_TEMPLATE_SHARE.get(level, 1.0)))
        for tid, n in counts.items():
            if n > share_limit:
                report["warnings"].append(f"template_overused:{level}:{tid}:{n}>{share_limit}")
    report_path = path.with_suffix(".validation_report.json")
    with report_path.open("w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)
    print(json.dumps(report, ensure_ascii=False, indent=2)[:5000])
    print(f"✅ validation report saved to {report_path}")
    return report

print("✅ v10 overrides active: strict gold quality, no country-year-only L2, deterministic L3-L5 candidate banks, L5 hard validation")


✅ v10 overrides active: strict gold quality, no country-year-only L2, deterministic L3-L5 candidate banks, L5 hard validation


In [34]:

# ============================================================
# 13c. v11 overrides: fast-fail WDQS + prioritized candidate banks + heartbeat
# ============================================================
# v10 could look frozen because the first shuffled candidate could be a heavy
# P108/P166 staff-award query and WDQS would retry it for minutes without any
# per-candidate heartbeat. v11 changes generation mechanics only: schema,
# constraints, gold collection and validators remain strict.

VERSION = "v11"
MAX_ATTEMPTS_PER_LEVEL = 500
MAX_CANDIDATES_PER_LEVEL = {"L1": 0, "L2": 90, "L3": 140, "L4": 160, "L5": 180}
CANDIDATE_HEARTBEAT_EVERY = 5
SLOW_CANDIDATE_SECONDS = 12.0

# Fail slow WDQS candidates quickly instead of spending 4 retry cycles on one
# bad candidate. The final JSONL is still validated strictly, so failed
# candidates are simply skipped.
try:
    WD_CLIENT.timeout = min(int(getattr(WD_CLIENT, "timeout", 18) or 18), 18)
    WD_CLIENT.max_retries = min(int(getattr(WD_CLIENT, "max_retries", 1) or 1), 1)
    WD_CLIENT.min_delay = min(float(getattr(WD_CLIENT, "min_delay", 0.20) or 0.20), 0.20)
except Exception:
    pass

PREFERRED_COUNTRY_QIDS = ["Q30", "Q145", "Q17", "Q258", "Q36", "Q40", "Q55", "Q183", "Q142", "Q96", "Q35", "Q33"]
PREFERRED_OCC_EN = ["physicist", "mathematician", "economist", "politician", "writer", "engineer", "computer scientist"]
PREFERRED_FOUNDER_EN = ["scientist", "writer", "politician", "engineer", "physicist"]
PREFERRED_AWARD_EN = ["Nobel Prize", "Turing Award", "Fields Medal"]

def _qid_country_list(qids):
    out = []
    for qid in qids:
        try:
            out.append(country_by_qid(qid))
        except Exception:
            continue
    return out

def _occ_list(names, pool=None):
    out = []
    pool = pool or STRICT_OCCUPATIONS
    for name in names:
        try:
            out.append(occ_by_en(name, pool=pool))
        except Exception:
            continue
    return out

def _award_list(names):
    out = []
    for name in names:
        try:
            out.append(award_by_en(name))
        except Exception:
            continue
    return out

def _short_desc(builder, kwargs):
    bits = [getattr(builder, "__name__", str(builder)).replace("make_", "")]
    for key, val in kwargs.items():
        if isinstance(val, dict):
            bits.append(f"{key}={val.get('en') or val.get('country_en') or val.get('capital_en') or val.get('qid')}")
        elif isinstance(val, tuple):
            bits.append(f"{key}={val[0]}-{val[1]}")
        else:
            bits.append(f"{key}={val}")
    return " | ".join(bits)

def _bind(builder: Callable[..., Optional[BenchmarkExample]], **kwargs: Any) -> Callable[[int, random.Random], Optional[BenchmarkExample]]:
    def _candidate(idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
        return builder(idx, rng, **kwargs)
    _candidate.__name__ = getattr(builder, "__name__", "candidate")
    _candidate._desc = _short_desc(builder, kwargs)
    return _candidate

def build_candidate_bank(level: str, rng: random.Random) -> List[Callable[[int, random.Random], Optional[BenchmarkExample]]]:
    """Prioritized v11 bank.

    Unlike v10, this does not shuffle heavy staff templates to the front.
    Fast/high-yield alumnus/founder patterns are tried first; staff-heavy
    patterns remain as later fallback candidates.
    """
    bank: List[Callable[[int, random.Random], Optional[BenchmarkExample]]] = []
    countries = _qid_country_list(PREFERRED_COUNTRY_QIDS)
    occs = _occ_list(PREFERRED_OCC_EN)
    founders = _occ_list(PREFERRED_FOUNDER_EN, pool=FOUNDER_OCCUPATIONS)
    awards = _award_list(PREFERRED_AWARD_EN)
    capitals = [c for c in CAPITAL_COUNTRIES if c.get("country_qid") in set(PREFERRED_COUNTRY_QIDS)]
    anchors = [a for a in STATIC_ANCHORS if a.get("country_qid") in set(PREFERRED_COUNTRY_QIDS)]

    if level == "L2":
        # Fast first: P69/P166 alumnus-award queries are usually much faster
        # and cleaner than P108/P166 staff-award queries.
        for c in countries:
            for a in awards:
                bank.append(_bind(make_l2_country_alumnus_award_v10, country=c, award=a))
        for c in countries:
            for fo in founders:
                bank.append(_bind(make_l2_country_founder_occupation_v10, country=c, occ=fo))
        # Heavy fallback only after all fast candidates.
        for c in countries[:6]:
            for a in awards:
                bank.append(_bind(make_l2_country_staff_award_v10, country=c, award=a))

    elif level == "L3":
        # Avoid broad staff_occupation anchor queries in the primary bank; they
        # caused huge noisy result sets and slow generation.
        for c in countries:
            for o in occs:
                for w in YEAR_WINDOWS_L3:
                    bank.append(_bind(make_l3_country_year_alumnus_occupation_v10, country=c, occ=o, window=w))
        for c in countries:
            for fo in founders:
                for ao in occs:
                    if fo.get("qid") != ao.get("qid"):
                        bank.append(_bind(make_l3_country_founder_alumnus_occupation_v10, country=c, founder_occ=fo, alumnus_occ=ao))
        for cc in capitals:
            for a in awards:
                for w in YEAR_WINDOWS_L3:
                    bank.append(_bind(make_l3_capital_year_alumnus_award_v10, country=cc, award=a, window=w))

    elif level == "L4":
        for c in countries:
            for fo in founders:
                for a in awards:
                    bank.append(_bind(make_l4_country_founder_alumnus_award_v10, country=c, founder_occ=fo, award=a))
        for anchor in anchors:
            for a in awards:
                bank.append(_bind(make_l4_same_country_after_anchor_alumnus_award_v10, anchor=anchor, award=a))
        for cc in capitals:
            for o in occs:
                for w in YEAR_WINDOWS_L4:
                    bank.append(_bind(make_l4_capital_year_alumnus_occupation_v10, country=cc, occ=o, window=w))
        # Staff-award fallback last.
        for c in countries[:8]:
            for ao in occs:
                for a in awards:
                    bank.append(_bind(make_l4_country_alumnus_occupation_staff_award_v10, country=c, occ=ao, award=a))

    elif level == "L5":
        # L5 must have 4 semantic criteria, but try the less explosive founder +
        # alumnus-award + alumnus-occupation family first.
        for c in countries:
            for fo in founders:
                for a in awards:
                    for ao in occs:
                        if fo.get("qid") != ao.get("qid"):
                            bank.append(_bind(make_l5_country_founder_alumnus_award_occupation_v10, country=c, founder_occ=fo, award=a, alumnus_occ=ao))
        for cc in capitals:
            for a in awards:
                for so in [o for o in occs if o.get("en") in {"physicist", "mathematician", "engineer", "computer scientist"}]:
                    for w in YEAR_WINDOWS_L5:
                        bank.append(_bind(make_l5_capital_year_award_staff_occupation_v10, country=cc, award=a, staff_occ=so, window=w))
        for anchor in anchors:
            for aa in awards:
                for sa in awards:
                    bank.append(_bind(make_l5_same_country_after_anchor_award_staff_award_v10, anchor=anchor, alumnus_award=aa, staff_award=sa))
        for c in countries[:8]:
            for aa in awards:
                for sa in awards:
                    for o in occs:
                        bank.append(_bind(make_l5_country_alumnus_staff_award_occupation_v10, country=c, alumnus_award=aa, staff_award=sa, occ=o))

    # Keep deterministic order, but rotate by seed so reruns are not identical.
    limit = int(MAX_CANDIDATES_PER_LEVEL.get(level, len(bank)))
    if bank and SEED is not None:
        shift = int(SEED) % len(bank)
        bank = bank[shift:] + bank[:shift]
    return bank[:limit]


def _candidate_desc(candidate: Callable[..., Any]) -> str:
    return str(getattr(candidate, "_desc", getattr(candidate, "__name__", "candidate")))[:180]


def generate_universities_dataset(
    *,
    out_path: Path = OUTPUT_PATH,
    target_per_level: Dict[str, int] = TARGET_PER_LEVEL,
    seed: int = SEED,
    max_attempts_per_level: int = MAX_ATTEMPTS_PER_LEVEL,
    overwrite: bool = OVERWRITE_OUTPUT,
) -> List[BenchmarkExample]:
    rng = random.Random(seed)
    if overwrite and out_path.exists():
        out_path.unlink()
    examples: List[BenchmarkExample] = []
    seen_semantic = set()
    seen_gold = set()
    reject_counts: Counter = Counter()
    template_counts_by_level: Dict[str, Counter] = {level: Counter() for level in LEVELS}

    for level in LEVELS:
        target = int(target_per_level.get(level, 0))
        if target <= 0:
            continue
        accepted = 0
        attempts = 0
        bank = build_candidate_bank(level, rng)
        print(f"[{DOMAIN}] {level}: candidate bank={len(bank)} target={target}; WDQS timeout={getattr(WD_CLIENT, 'timeout', '?')}s retries={getattr(WD_CLIENT, 'max_retries', '?')}")

        for candidate in bank:
            if accepted >= target:
                break
            attempts += 1
            idx = accepted + 1
            global LAST_REJECT_REASON
            LAST_REJECT_REASON = "none"
            desc = _candidate_desc(candidate)
            if attempts == 1 or attempts % CANDIDATE_HEARTBEAT_EVERY == 0:
                print(f"[{DOMAIN}] {level}: trying {attempts}/{len(bank)}; accepted={accepted}/{target}; {desc}")
            t0 = time.perf_counter()
            try:
                ex = candidate(idx, rng)
            except Exception as e:
                elapsed = time.perf_counter() - t0
                reject_counts[(level, "exception", str(e)[:100])] += 1
                print(f"[{DOMAIN}] {level}: reject exception after {elapsed:.1f}s — {desc} — {str(e)[:120]}")
                continue
            elapsed = time.perf_counter() - t0
            if elapsed >= SLOW_CANDIDATE_SECONDS:
                print(f"[{DOMAIN}] {level}: slow candidate {elapsed:.1f}s — {desc} — result={'ok' if ex else 'reject'}")
            if ex is None:
                reject_counts[(level, LAST_REJECT_REASON or "none")] += 1
                continue
            errors = validate_example_basic(ex)
            if errors:
                reject_counts[(level, ",".join(errors[:2]))] += 1
                continue
            share_limit = max(1, int(target * MAX_TEMPLATE_SHARE.get(level, 1.0)))
            if template_counts_by_level[level][ex.template_id] >= share_limit:
                reject_counts[(level, "template_quota", ex.template_id)] += 1
                continue
            sk = semantic_key(ex)
            gk = gold_key(ex)
            if sk in seen_semantic:
                reject_counts[(level, "duplicate_semantic")] += 1
                continue
            if gk in seen_gold:
                reject_counts[(level, "duplicate_gold_set")] += 1
                continue
            seen_semantic.add(sk)
            seen_gold.add(gk)
            template_counts_by_level[level][ex.template_id] += 1
            examples.append(ex)
            accepted += 1
            append_example_jsonl(out_path, ex)
            print(f"[{DOMAIN}] {level}: {accepted}/{target} saved — {ex.template_id} — gold={len(ex.gold_answer_qids)} — {elapsed:.1f}s")

        # Very short fallback, also with heartbeat. If this cannot finish, final
        # validation fails loudly instead of hiding the issue.
        fallback_attempts = 0
        while accepted < target and fallback_attempts < max_attempts_per_level:
            fallback_attempts += 1
            attempts += 1
            idx = accepted + 1
            LAST_REJECT_REASON = "none"
            if fallback_attempts == 1 or fallback_attempts % 25 == 0:
                print(f"[{DOMAIN}] {level}: fallback {fallback_attempts}/{max_attempts_per_level}; accepted={accepted}/{target}")
            t0 = time.perf_counter()
            try:
                ex = generate_one(level, idx, rng)
            except Exception as e:
                reject_counts[(level, "fallback_exception", str(e)[:100])] += 1
                continue
            elapsed = time.perf_counter() - t0
            if ex is None:
                reject_counts[(level, LAST_REJECT_REASON or "none")] += 1
                continue
            errors = validate_example_basic(ex)
            if errors:
                reject_counts[(level, ",".join(errors[:2]))] += 1
                continue
            share_limit = max(1, int(target * MAX_TEMPLATE_SHARE.get(level, 1.0)))
            if template_counts_by_level[level][ex.template_id] >= share_limit:
                reject_counts[(level, "template_quota", ex.template_id)] += 1
                continue
            sk = semantic_key(ex); gk = gold_key(ex)
            if sk in seen_semantic or gk in seen_gold:
                reject_counts[(level, "duplicate")] += 1
                continue
            seen_semantic.add(sk); seen_gold.add(gk)
            template_counts_by_level[level][ex.template_id] += 1
            examples.append(ex); accepted += 1
            append_example_jsonl(out_path, ex)
            print(f"[{DOMAIN}] {level}: {accepted}/{target} saved — {ex.template_id} — gold={len(ex.gold_answer_qids)} — {elapsed:.1f}s [fallback]")

        if accepted < target:
            print(f"[WARN] {DOMAIN}:{level} generated {accepted}/{target} after {attempts} attempts")
            print("[WARN] common reject reasons:", reject_counts.most_common(15))
        else:
            print(f"✅ {DOMAIN}:{level} complete in {attempts} attempts; templates={dict(template_counts_by_level[level])}")
    print(f"✅ generation finished: {len(examples)} examples saved to {out_path}")
    return examples

print("✅ v11 overrides active: fast WDQS timeout, prioritized fast candidate banks, visible candidate heartbeat")


✅ v11 overrides active: fast WDQS timeout, prioritized fast candidate banks, visible candidate heartbeat


In [35]:
# ============================================================
# 14. Run generation


In [ ]:
# ============================================================
if RUN_GENERATION:
    examples = generate_universities_dataset(
        out_path=OUTPUT_PATH,
        target_per_level=TARGET_PER_LEVEL,
        seed=SEED,
        max_attempts_per_level=MAX_ATTEMPTS_PER_LEVEL,
        overwrite=OVERWRITE_OUTPUT,
    )
    if RUN_FINAL_VALIDATION:
        validation_report = validate_output_file(OUTPUT_PATH)
        if validation_report.get("errors"):
            raise RuntimeError(f"Validation failed with {len(validation_report['errors'])} errors. See {OUTPUT_PATH.with_suffix('.validation_report.json')}")
        show_sample(OUTPUT_PATH, n=5)
else:
    print("RUN_GENERATION=False; set it to True to build universities.jsonl")


[universities] L2: candidate bank=90 target=15; WDQS timeout=18s retries=1
[universities] L2: trying 1/90; accepted=0/15; l2_country_founder_occupation_v10 | country=France | occ=politician
